# Proyecto final · Demanda laboral en Panamá: qué piden y qué ofrecen las empresas en Konzerta
**Introducción a la Programación para Ciencia de Datos** · Maestría en Ciencias de los Datos · UIP<br>
Profesor: Mgtr. Jose Higuera

| Campo | Dato |
|---|---|
| Estudiante: | Simón Ali Bustamante Blanco |
| Pasaporte: | 163511544 |
| Modalidad: | Individual |
| Presentación: | 07-11-2026 |

Este cuaderno contiene todo el flujo del proyecto, de los datos crudos al dashboard:

1. Configuración y lectura de los datos originales
2. Documentación del dataset (fuente y diccionario de datos)
3. Perfil antes de la limpieza
4. Limpieza y bitácora
5. Perfil después de la limpieza
6. Tablas de análisis
7. Análisis exploratorio (EDA)
8. Dashboard (`reporte/dashboard/index.html`)

Para regenerar todo: **Restart & Run All**.

## 1. Configuración y lectura de los datos originales
El cuaderno toma automáticamente el Excel de Konzerta **más reciente** de `datos/originales/`. Así, si se agrega una nueva extracción, basta con volver a ejecutar sin editar el código.

El archivo original solo se lee: se guarda en `crudos` y toda la limpieza se hace sobre la copia `df`.

In [284]:
# Autor: Simón Ali Bustamante Blanco · Proyecto final IPCD · UIP · 2026
# Librerías
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)

# Rutas del proyecto (funciona tanto desde notebooks/ como desde la raíz)
BASE = Path.cwd() if (Path.cwd() / "datos").exists() else Path.cwd().parent
DIR_ORIGINALES = BASE / "datos" / "originales"
DIR_LIMPIOS = BASE / "datos" / "limpios"
DIR_DASHBOARD = BASE / "reporte" / "dashboard"

# Archivo de Konzerta más reciente (se ignoran los temporales de Excel "~$...")
archivos = [a for a in DIR_ORIGINALES.glob("Konzerta*.xlsx") if not a.name.startswith("~$")]
ARCHIVO_ORIGINAL = max(archivos, key=lambda a: a.stat().st_mtime)

crudos = pd.read_excel(ARCHIVO_ORIGINAL)   # datos originales: no se modifican
df = crudos.copy()                         # copia sobre la que se hace la limpieza

print("Archivo leído:", ARCHIVO_ORIGINAL.name)
print("Tamaño (filas, columnas):", crudos.shape)

Archivo leído: Konzerta 08-oct-2026.xlsx
Tamaño (filas, columnas): (3011, 31)


## 2. Documentación del dataset

| Campo | Dato |
|---|---|
| **Qué son** | Ofertas de empleo publicadas en el portal Konzerta.com para Panamá. Cada fila es un aviso. |
| **Fuente** | Dataset proporcionado por el profesor, obtenido de los avisos públicos de [konzerta.com](https://www.konzerta.com) |
| **Fecha de obtención** | 08/10/2026 (columna `extraido`) |
| **Formato** | Excel (.xlsx), una hoja |
| **Confidencialidad** | Datos públicos. Aun así, no se publican las URL ni las descripciones completas, y las empresas confidenciales se muestran como «Confidencial». |

El tamaño y el periodo cubierto se calculan desde los datos:

In [285]:
# Tamaño y periodo cubierto, calculados desde los datos
inicio = crudos["fecha_publicacion"].min()
fin = crudos["fecha_publicacion"].max()
extraccion = pd.to_datetime(crudos["extraido"]).max()

print(f"Registros: {crudos.shape[0]:,} avisos · Columnas: {crudos.shape[1]}")
print(f"Periodo de publicación: {inicio:%d/%m/%Y} – {fin:%d/%m/%Y} ({(fin - inicio).days} días)")
print(f"Fecha de extracción: {extraccion:%d/%m/%Y}")

Registros: 3,011 avisos · Columnas: 31
Periodo de publicación: 11/07/2026 – 08/10/2026 (89 días)
Fecha de extracción: 08/10/2026


### 2.1 Pertinencia: por qué este dataset
**Mercado de estudio:** el mercado laboral panameño, visto desde la oferta de empleo formal en línea. Permite medir con datos qué perfiles buscan las empresas, en qué regiones y bajo qué condiciones, en lugar de suponerlo.

**A quién le sirve:**
- **Personas que buscan empleo:** qué áreas tienen más demanda y qué requisitos priorizar.
- **Universidades como la UIP:** alinear su oferta académica con lo que pide el mercado.
- **Equipos de Recursos Humanos:** comparar sus salarios y requisitos con el mercado.
- **Entidades públicas:** conocer la concentración geográfica del empleo.

**Preguntas del análisis** (de la propuesta):

| # | Pregunta | Columnas que la responden |
|---|---|---|
| 1 | ¿Qué áreas, industrias y provincias concentran la demanda de empleo? | `area`, `industria`, `provincia`, `vacantes` |
| 2 | ¿Cuánto se ofrece de salario según el nivel laboral y el área, y qué tan común es publicarlo? | `salario_min`, `salario_max`, `nivel_laboral`, `area` |
| 3 | ¿Qué requisitos se piden por nivel: educación, experiencia, idiomas y rango de edad? | `educacion`, `experiencia_meses`, `idiomas`, `edad_min`, `edad_max` |
| 4 | ¿Cómo evoluciona la publicación semanal de avisos y qué peso tienen el trabajo remoto e híbrido? | `fecha_publicacion`, `modalidad` |

**Reporte:** opción B, dashboard web en HTML (`reporte/dashboard/index.html`), generado por la sección 8 de este cuaderno.

### 2.2 Diccionario de datos
Las descripciones y los tipos se escriben a mano. El **tipo actual**, los **valores distintos** y el **% de faltantes** se calculan desde los datos, así el diccionario se actualiza si cambia el archivo.

- **Tipo actual:** cómo lo leyó pandas.
- **Tipo final:** el tipo que tendrá después de la limpieza.

Las columnas con pocos valores que se repiten pasan a `category`, el tipo de pandas para variables categóricas. Las **ordinales** (`nivel_laboral`, `empresa_tamano`, `educacion`) serán `category` **ordenada**, para que tablas y gráficos respeten su orden natural (p. ej. Junior → Semi Sr → Senior) y no el alfabético. El CSV no guarda este tipo, así que el orden se vuelve a aplicar al leerlo.

In [286]:
# Descripción, tipo de variable y tipo final de cada columna
descripciones = {
    "id":                   ("Identificador", "int", "Identificador único del aviso en Konzerta"),
    "titulo":               ("Texto", "str", "Título del puesto publicado"),
    "empresa":              ("Categórica", "category", "Nombre de la empresa (vacío cuando el aviso es confidencial)"),
    "empresa_confidencial": ("Booleana", "bool", "True si la empresa ocultó su nombre"),
    "industria":            ("Categórica", "category", "Sector económico de la empresa"),
    "empresa_tamano":       ("Categórica ordinal", "category (ordenada)", "Rango de número de empleados de la empresa"),
    "fecha_publicacion":    ("Fecha", "datetime64", "Fecha en que se publicó el aviso"),
    "fecha_finalizacion":   ("Fecha", "datetime64", "Fecha en que vence el aviso"),
    "provincia":            ("Categórica", "category", "Provincia donde está el puesto"),
    "localidad":            ("Categórica", "category", "Ciudad o corregimiento del puesto (texto libre, con variantes)"),
    "area":                 ("Categórica", "category", "Área funcional del puesto"),
    "subarea":              ("Categórica", "category", "Sub-área funcional dentro del área"),
    "nivel_laboral":        ("Categórica ordinal", "category (ordenada)", "Nivel jerárquico (Junior, Semi Sr, Senior, Jefe, Gerencia…)"),
    "tipo_trabajo":         ("Categórica", "category", "Jornada (Full-time, Part-time, Por horas…)"),
    "modalidad":            ("Categórica", "category", "Presencial, Híbrido o Remoto"),
    "contratacion":         ("Categórica", "category", "Tipo de contrato (Indeterminado, Temporal, Pasantía…)"),
    "vacantes":             ("Numérica", "int", "Número de plazas del aviso"),
    "salario_min":          ("Numérica", "float", "Salario mínimo ofrecido, B/. mensuales brutos"),
    "salario_max":          ("Numérica", "float", "Salario máximo ofrecido, B/. mensuales brutos"),
    "salario_frecuencia":   ("Categórica", "— (se elimina)", "Periodicidad del salario (un solo valor: «por Mes»)"),
    "salario_tipo":         ("Categórica", "— (se elimina)", "Bruto o neto (un solo valor: «BRUTO»)"),
    "edad_min":             ("Numérica", "float", "Edad mínima pedida, en años"),
    "edad_max":             ("Numérica", "float", "Edad máxima pedida, en años"),
    "experiencia_meses":    ("Numérica", "float", "Experiencia mínima pedida, en meses"),
    "educacion":            ("Categórica ordinal", "category (ordenada)", "Nivel educativo mínimo (Secundario, Técnico, Universitario, Posgrado…)"),
    "educacion_estado":     ("Categórica", "category", "Estado del estudio pedido (Graduado o En Curso)"),
    "idiomas":              ("Texto", "str → columnas", "Idiomas y nivel en un solo texto, separados por «;» (p. ej. «Inglés (Avanzado)»)"),
    "apto_discapacidad":    ("Booleana", "bool", "True si el aviso es apto para personas con discapacidad"),
    "descripcion":          ("Texto", "— (no se publica)", "Texto completo del aviso"),
    "url":                  ("Texto", "— (no se publica)", "Enlace al aviso en Konzerta"),
    "extraido":             ("Fecha", "datetime64", "Fecha y hora en que se extrajo el aviso"),
}

# Columnas nuevas que no estén en el diccionario quedan marcadas para documentarlas
diccionario = pd.DataFrame({
    "columna": crudos.columns,
    "variable": [descripciones.get(c, ("Sin documentar",) * 3)[0] for c in crudos.columns],  # tipo de variable (1.er dato de la tupla); «Sin documentar» si la columna no está en el diccionario
    "tipo_actual": crudos.dtypes.astype(str).values,  # tipo con que pandas leyó cada columna, como texto
    "tipo_final": [descripciones.get(c, ("Sin documentar",) * 3)[1] for c in crudos.columns],  # tipo que tendrá tras la limpieza (2.º dato de la tupla)
    "valores_distintos": crudos.nunique().values,  # cantidad de valores distintos por columna (sin contar vacíos)
    "pct_faltantes": (crudos.isna().mean() * 100).round(1).values,  # % de filas vacías por columna, con 1 decimal
    "descripcion": [descripciones.get(c, ("Sin documentar",) * 3)[2] for c in crudos.columns],  # descripción escrita a mano (3.er dato de la tupla)
})

sin_documentar = diccionario.loc[diccionario["variable"] == "Sin documentar", "columna"].tolist()
print("Columnas sin documentar:", sin_documentar or "ninguna")
diccionario

Columnas sin documentar: ninguna


,columna,variable,tipo_actual,tipo_final,valores_distintos,pct_faltantes,descripcion
0,id,Identificador,int64,int,3011,0.0,Identificador único del aviso en Konzerta
1,titulo,Texto,str,str,2629,0.0,Título del puesto publicado
2,empresa,Categórica,str,category,635,38.7,Nombre de la empresa (vacío cuando el aviso es confidencial)
3,empresa_confidencial,Booleana,bool,bool,2,0.0,True si la empresa ocultó su nombre
4,industria,Categórica,str,category,54,40.6,Sector económico de la empresa
5,empresa_tamano,Categórica ordinal,str,category (ordenada),7,39.6,Rango de número de empleados de la empresa
6,fecha_publicacion,Fecha,datetime64[us],datetime64,87,0.0,Fecha en que se publicó el aviso
7,fecha_finalizacion,Fecha,datetime64[us],datetime64,87,0.0,Fecha en que vence el aviso
8,provincia,Categórica,str,category,11,0.0,Provincia donde está el puesto
9,localidad,Categórica,str,category,104,1.4,"Ciudad o corregimiento del puesto (texto libre, con variantes)"


In [287]:
# Resumen por tipo de variable: confirma que hay numéricas, categóricas y fechas
diccionario["variable"].value_counts()

variable
Categórica            12
Numérica               6
Texto                  4
Categórica ordinal     3
Fecha                  3
Booleana               2
Identificador          1
Name: count, dtype: int64

### 2.3 Muestra del dataset
Primeras filas con columnas seleccionadas. No se muestran `url` ni `descripcion`.

In [288]:
columnas_muestra = ["titulo", "empresa", "area", "provincia", "nivel_laboral",
                    "modalidad", "salario_min", "salario_max", "fecha_publicacion"]
crudos[columnas_muestra].head(6)

,titulo,empresa,area,provincia,nivel_laboral,modalidad,salario_min,salario_max,fecha_publicacion
0,Ortodoncista,Life Time Smile,"Salud, Medicina y Farmacia",Panamá,NaN,Presencial,1500.0,1800.0,2026-10-08
1,Gerente de Auditoría Externa,NaN,"Administración, Contabilidad y Finanzas",Panamá,Gerencia / Alta Gerencia / Dirección,Presencial,NaN,NaN,2026-10-07
2,Ejecutivo de Atencion al Cliente - Canales Digitales,NaN,"Atención al Cliente, Call Center y Telemarketing",Panamá,Junior,Presencial,NaN,NaN,2026-10-07
3,CAJERO y VENDEDOR - DAVID - SAN MATEO,NaN,"Comercial, Ventas y Negocios",Chiriquí,Otro,Presencial,NaN,NaN,2026-10-07
4,Técnico en Reparación de Herramientas Eléctricas-Combustión,Empresas Carbone S.A,Ingenierías,Panamá,Semi Sr,Presencial,NaN,NaN,2026-10-07
5,Asesor de Venta Optica Optikal Multiplaza,Óptica Optikal,"Comercial, Ventas y Negocios",Panamá,Junior,Presencial,NaN,NaN,2026-10-07


### 2.4 Por qué requiere limpieza
El dataset **no viene depurado**. Las celdas siguientes miden sus problemas de calidad directamente sobre los datos originales; cada uno se trata y se justifica en la sección 4 (limpieza y bitácora).

**Faltantes por columna** (solo las que tienen alguno):

In [289]:
# Faltantes por columna, de mayor a menor
faltantes = pd.DataFrame({
    "faltantes": crudos.isna().sum(),
    "pct_faltantes": (crudos.isna().mean() * 100).round(1),
})
faltantes[faltantes["faltantes"] > 0].sort_values("faltantes", ascending=False)

,faltantes,pct_faltantes
idiomas,2606,86.5
salario_min,2546,84.6
salario_frecuencia,2526,83.9
salario_max,2526,83.9
salario_tipo,2526,83.9
edad_min,2344,77.8
edad_max,2344,77.8
experiencia_meses,2159,71.7
educacion,1789,59.4
educacion_estado,1789,59.4


**Otros problemas de calidad:** atípicos, formatos, categorías equivalentes, columnas constantes y duplicados.

In [290]:
def contar(columna, valor):
    """Cuántas filas tienen exactamente ese valor en la columna."""
    return int((crudos[columna] == valor).sum())

titulos = crudos["titulo"].str.strip()
en_mayusculas = (titulos == titulos.str.upper()) & titulos.str.contains("[A-Za-z]")
constantes = [c for c in crudos.columns if crudos[c].nunique() == 1]
sin_identificadores = crudos.drop(columns=["id", "url", "extraido"])

problemas = pd.DataFrame([
    ("Atípicos", "salario_max", f"{(crudos['salario_max'] < 100).sum()} salarios máximos menores a B/. 100"),
    ("Atípicos", "salario_max", f"Máximo de B/. {crudos['salario_max'].max():,.0f} mensuales"),
    ("Atípicos", "edad_max", f"{(crudos['edad_max'] > 65).sum()} avisos con edad máxima mayor a 65 (máximo {crudos['edad_max'].max():.0f})"),
    ("Formato", "titulo", f"{en_mayusculas.mean():.0%} de los títulos en MAYÚSCULAS ({en_mayusculas.sum()} avisos)"),
    ("Formato", "localidad", f"Sin tilde: «Chiriqui» ({contar('localidad', 'Chiriqui')}), «Cocle» ({contar('localidad', 'Cocle')})"),
    ("Equivalentes", "localidad", f"«Betania» ({contar('localidad', 'Betania')}) / «Bethania» ({contar('localidad', 'Bethania')})"),
    ("Equivalentes", "localidad", f"«La Chorrera» ({contar('localidad', 'La Chorrera')}) / «Chorrera» ({contar('localidad', 'Chorrera')})"),
    ("Equivalentes", "localidad", f"«Ciudad De Panama» ({contar('localidad', 'Ciudad De Panama')}) / «Panama City» ({contar('localidad', 'Panama City')}) / «Panamá» ({contar('localidad', 'Panamá')})"),
    ("Equivalentes", "localidad", f"«Tocumen.» ({contar('localidad', 'Tocumen.')}) / «Tocume  Panama» ({contar('localidad', 'Tocume  Panama')})"),
    ("Equivalentes", "nivel_laboral", f"«Senior / Semi-Senior» ({contar('nivel_laboral', 'Senior / Semi-Senior')}) se solapa con «Senior» y «Semi Sr»"),
    ("Equivalentes", "industria", f"«Banca / Financiera» ({contar('industria', 'Banca / Financiera')}) / «Financiera» ({contar('industria', 'Financiera')})"),
    ("Texto compuesto", "idiomas", f"{crudos['idiomas'].str.contains(';', na=False).sum()} avisos con varios idiomas en un solo texto (p. ej. «Español (Nativo); Inglés (Básico)»)"),
    ("Constantes", ", ".join(constantes), "Un solo valor en toda la columna: no aportan información"),
    ("Duplicados", "todas", f"{crudos.duplicated().sum()} filas idénticas"),
    ("Duplicados", "todas menos id, url, extraido", f"{sin_identificadores.duplicated().sum()} avisos repetidos con distinto id"),
    ("Duplicados", "titulo + empresa + provincia", f"{crudos.duplicated(['titulo', 'empresa', 'provincia']).sum()} posibles republicaciones"),
    ("Duplicados", "descripcion", f"{crudos['descripcion'].duplicated().sum()} descripciones repetidas"),
    ("Sensible", "empresa", f"{crudos['empresa_confidencial'].sum()} avisos confidenciales (empresa vacía: {crudos['empresa'].isna().sum()})"),
    ("Sensible", "url, descripcion", "Enlace y texto completo del aviso: no se publican"),
], columns=["problema", "columna", "evidencia"])
problemas

,problema,columna,evidencia
0,Atípicos,salario_max,31 salarios máximos menores a B/. 100
1,Atípicos,salario_max,"Máximo de B/. 1,350,000 mensuales"
2,Atípicos,edad_max,3 avisos con edad máxima mayor a 65 (máximo 99)
3,Formato,titulo,25% de los títulos en MAYÚSCULAS (748 avisos)
4,Formato,localidad,"Sin tilde: «Chiriqui» (23), «Cocle» (3)"
5,Equivalentes,localidad,«Betania» (59) / «Bethania» (42)
6,Equivalentes,localidad,«La Chorrera» (42) / «Chorrera» (4)
7,Equivalentes,localidad,«Ciudad De Panama» (996) / «Panama City» (202) / «Panamá» (515)
8,Equivalentes,localidad,«Tocumen.» (51) / «Tocume Panama» (14)
9,Equivalentes,nivel_laboral,«Senior / Semi-Senior» (193) se solapa con «Senior» y «Semi Sr»


**Conclusión:** el dataset es adecuado para el proyecto. Supera los 1,000 registros, combina variables numéricas, categóricas, booleanas y fechas, y presenta problemas reales de calidad (faltantes, atípicos, formatos, categorías equivalentes, columnas constantes y posibles duplicados) que justifican la limpieza de la sección 4.

## 3. Perfil antes de la limpieza
Foto del dataset original: tamaño, tipos, faltantes y duplicados. Las mismas funciones se usan en la sección 5 para comparar el antes y el después.

In [291]:
def perfil(datos):
    """Perfil por columna: tipo, valores no nulos, faltantes y valores distintos."""
    return pd.DataFrame({
        "tipo": datos.dtypes.astype(str),
        "no_nulos": datos.notna().sum(),
        "faltantes": datos.isna().sum(),
        "pct_faltantes": (datos.isna().mean() * 100).round(1),
        "valores_distintos": datos.nunique(),
    })

def resumen(datos):
    """Resumen general: filas, columnas, celdas vacías, columnas con faltantes y filas duplicadas."""
    return pd.Series({
        "filas": datos.shape[0],
        "columnas": datos.shape[1],
        "celdas_vacias": int(datos.isna().sum().sum()),
        "pct_celdas_vacias": round(datos.isna().mean().mean() * 100, 1),
        "columnas_con_faltantes": int((datos.isna().sum() > 0).sum()),
        "filas_duplicadas": int(datos.duplicated().sum()),
    }, dtype="object")   # object: los conteos se muestran como enteros

perfil_antes = perfil(crudos)
resumen_antes = resumen(crudos)

print(resumen_antes.to_string())
perfil_antes

filas                      3011
columnas                     31
celdas_vacias             27627
pct_celdas_vacias          29.6
columnas_con_faltantes       16
filas_duplicadas              0


,tipo,no_nulos,faltantes,pct_faltantes,valores_distintos
id,int64,3011,0,0.0,3011
titulo,str,3011,0,0.0,2629
empresa,str,1846,1165,38.7,635
empresa_confidencial,bool,3011,0,0.0,2
industria,str,1788,1223,40.6,54
empresa_tamano,str,1820,1191,39.6,7
fecha_publicacion,datetime64[us],3011,0,0.0,87
fecha_finalizacion,datetime64[us],3011,0,0.0,87
provincia,str,3011,0,0.0,11
localidad,str,2969,42,1.4,104


In [292]:
# Cantidad de columnas por tipo de dato, antes de limpiar
perfil_antes["tipo"].value_counts()

tipo
str               20
float64            5
int64              2
bool               2
datetime64[us]     2
Name: count, dtype: int64

## 4. Limpieza y bitácora
Toda la limpieza se hace sobre `df`, la copia de `crudos`. Cada transformación agrega una fila a la **bitácora** con la columna afectada y su justificación. La bitácora completa se muestra al final de esta sección.

Orden de los pasos:
1. Nombres de columnas
2. Tipos de datos
3. Textos estandarizados
4. Duplicados
5. Idiomas
6. Atípicos
7. Faltantes
8. Columnas constantes y sensibles
9. Tipo categoría

In [293]:
import re
from unidecode import unidecode
import plotly.express as px

registros = []   # filas de la bitácora: (Columna, Justificación)

def anotar(columna, justificacion):
    """Agrega una fila a la bitácora de limpieza."""
    registros.append((columna, justificacion))

### 4.1 Nombres de columnas
Se pasan a snake_case: minúsculas, sin tildes, sin espacios ni símbolos. El archivo actual ya viene así, pero la función protege el flujo si una nueva extracción trae otros nombres.

In [294]:
def a_snake_case(nombre):
    """Convierte un nombre de columna a snake_case: sin tildes, en minúsculas y con guion bajo."""
    nombre = unidecode(str(nombre)).strip().lower()
    return re.sub(r"[^a-z0-9]+", "_", nombre).strip("_")

nombres_antes = list(df.columns)
df.columns = [a_snake_case(c) for c in df.columns]
renombradas = [(a, b) for a, b in zip(nombres_antes, df.columns) if a != b]

print("Columnas renombradas:", renombradas or "ninguna, ya venían en snake_case")
anotar("todas", f"Nombres verificados en snake_case (minúsculas, sin tildes ni espacios); columnas renombradas: {len(renombradas)}")

Columnas renombradas: ninguna, ya venían en snake_case


### 4.2 Tipos de datos
Fechas a `datetime`, cantidades a número y banderas a booleano. Con `errors="coerce"`, un valor que no se puede leer queda vacío en lugar de detener el cuaderno. Las categorías se convierten al final (4.9), cuando sus valores ya están limpios.

In [295]:
columnas_fecha = ["fecha_publicacion", "fecha_finalizacion", "extraido"]
columnas_numericas = ["vacantes", "salario_min", "salario_max", "edad_min", "edad_max", "experiencia_meses"]
columnas_booleanas = ["empresa_confidencial", "apto_discapacidad"]

def a_booleano(serie):
    """Convierte una columna a booleano; acepta True/False o textos como «true», «sí», «no», «1», «0»."""
    if pd.api.types.is_bool_dtype(serie):
        return serie
    valores = {"true": True, "false": False, "si": True, "sí": True, "no": False, "1": True, "0": False}
    return serie.astype(str).str.strip().str.lower().map(valores).fillna(False).astype(bool)

tipos_antes = df.dtypes.astype(str)
for col in columnas_fecha:
    df[col] = pd.to_datetime(df[col], errors="coerce")
for col in columnas_numericas:
    df[col] = pd.to_numeric(df[col], errors="coerce")
for col in columnas_booleanas:
    df[col] = a_booleano(df[col])

anotar(", ".join(columnas_fecha), "Convertidas a datetime para analizar la evolución en el tiempo («extraido» venía como texto)")
anotar(", ".join(columnas_numericas), "Convertidas a número (errors='coerce') para calcular estadísticos")
anotar(", ".join(columnas_booleanas), "Aseguradas como booleanas (True/False)")

# Tipos antes y después de convertir
pd.DataFrame({"antes": tipos_antes, "despues": df.dtypes.astype(str)}).loc[columnas_fecha + columnas_numericas + columnas_booleanas]

,antes,despues
fecha_publicacion,datetime64[us],datetime64[us]
fecha_finalizacion,datetime64[us],datetime64[us]
extraido,str,datetime64[us]
vacantes,int64,int64
salario_min,float64,float64
salario_max,float64,float64
edad_min,float64,float64
edad_max,float64,float64
experiencia_meses,float64,float64
empresa_confidencial,bool,bool


### 4.3 Textos estandarizados
1. **Espacios:** se quitan los del inicio y el final, y los espacios dobles se reducen a uno.
2. **Títulos en MAYÚSCULAS:** se pasan a formato oración («ASESOR DE VENTAS» → «Asesor de ventas»).
3. **Valores equivalentes:** un mismo valor escrito de varias formas se unifica con un diccionario de reemplazo.

In [296]:
columnas_texto = [c for c in df.columns if pd.api.types.is_string_dtype(df[c])]

# 1. Espacios al inicio/final y espacios dobles
texto_antes = df[columnas_texto].copy()
for col in columnas_texto:
    df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)
corregidos = (texto_antes.fillna("") != df[columnas_texto].fillna("")).sum()
corregidos = corregidos[corregidos > 0]

print("Valores corregidos por espacios:")
print(corregidos.to_string())
anotar(", ".join(corregidos.index), f"Quitados espacios al inicio/final y espacios dobles ({corregidos.sum():,} valores)")

# 2. Títulos escritos completamente en MAYÚSCULAS → formato oración
en_mayusculas = (df["titulo"] == df["titulo"].str.upper()) & df["titulo"].str.contains("[A-Za-z]")
df.loc[en_mayusculas, "titulo"] = df.loc[en_mayusculas, "titulo"].str.capitalize()

print(f"\nTítulos pasados de MAYÚSCULAS a formato oración: {en_mayusculas.sum()}")
anotar("titulo", f"{en_mayusculas.sum()} títulos en MAYÚSCULAS pasados a formato oración para uniformar el texto")

Valores corregidos por espacios:
empresa         122
localidad        14
subarea          65
descripcion    3000

Títulos pasados de MAYÚSCULAS a formato oración: 748


In [297]:
# 3. Valores equivalentes: {columna: {valor_original: valor_unificado}}
reemplazos = {
    "localidad": {
        "Ciudad De Panama": "Ciudad de Panamá", "Panama City": "Ciudad de Panamá",
        "Betania": "Bethania",
        "Chorrera": "La Chorrera",
        "Tocumen.": "Tocumen", "Tocume Panama": "Tocumen", "Nuevo Tocumén": "Nuevo Tocumen",
        "San Miquelito": "San Miguelito",
        "David District": "David", "Santiago District": "Santiago",
        "Chiriqui": "Chiriquí", "Cocle": "Coclé",
        "Alcalde Diaz": "Alcalde Díaz", "Belen": "Belén", "San Felíx": "San Félix", "Isla Colon": "Isla Colón",
        "Sta. Maria": "Santa María", "Chanis Calle Principal.": "Chanis", "La Cabima.": "La Cabima",
    },
    "area": {"Departamento Tecnico": "Departamento Técnico", "Naviero, Maritimo, Portuario": "Naviero, Marítimo, Portuario"},
    "industria": {"Financiera": "Banca / Financiera"},
    "nivel_laboral": {"Senior / Semi-Senior": "Semi Sr"},
    "educacion": {"Master": "Maestría"},
    "tipo_trabajo": {"Pasantia": "Pasantía"},
}

# Experiencia mediana pedida por nivel: sustenta a qué nivel asignar «Senior / Semi-Senior»
experiencia_nivel = df.groupby("nivel_laboral")["experiencia_meses"].median()

motivos = {
    "localidad": "Unificadas variantes del mismo lugar (idioma, tildes, puntos, «District»), p. ej. «Panama City» → «Ciudad de Panamá», «Betania» → «Bethania»",
    "area": "Agregadas las tildes faltantes («Tecnico», «Maritimo»)",
    "industria": "«Financiera» y «Banca / Financiera» son el mismo sector",
    "nivel_laboral": (f"«Senior / Semi-Senior» acepta ambos niveles: se asigna al menor, «Semi Sr». "
                      f"Experiencia mediana pedida: {experiencia_nivel.get('Senior / Semi-Senior', float('nan')):.0f} meses, "
                      f"igual que Semi Sr ({experiencia_nivel.get('Semi Sr', float('nan')):.0f})"),
    "educacion": "«Master» pasa a «Maestría» para tener todas las categorías en español",
    "tipo_trabajo": "Agregada la tilde faltante («Pasantia»)",
}

for col, mapa in reemplazos.items():
    afectados = df[col].isin(mapa.keys()).sum()
    df[col] = df[col].replace(mapa)
    print(f"{col}: {afectados} valores unificados")
    anotar(col, f"{motivos[col]} ({afectados} valores)")

# «Panamá» como localidad es la ciudad solo cuando la provincia también es Panamá
ciudad = (df["localidad"] == "Panamá") & (df["provincia"] == "Panamá")
df.loc[ciudad, "localidad"] = "Ciudad de Panamá"
print(f"localidad: {ciudad.sum()} «Panamá» (provincia Panamá) → «Ciudad de Panamá»")
anotar("localidad", f"«Panamá» como localidad en la provincia de Panamá es la ciudad → «Ciudad de Panamá» ({ciudad.sum()} valores)")

localidad: 1375 valores unificados
area: 36 valores unificados
industria: 107 valores unificados
nivel_laboral: 193 valores unificados
educacion: 24 valores unificados
tipo_trabajo: 7 valores unificados
localidad: 514 «Panamá» (provincia Panamá) → «Ciudad de Panamá»


### 4.4 Duplicados
No hay filas idénticas, pero sí dos tipos de repetición:
1. **Mismo aviso cargado dos veces:** todo el contenido es igual (todas las columnas menos `id`, `url` y `extraido`). Se deja uno.
2. **Republicaciones:** mismo título, empresa, provincia y **descripción**, publicado en otra fecha. Es la misma vacante publicada de nuevo; contarla dos veces inflaría la demanda. Se deja la **primera publicación**.

Los avisos con igual título, empresa y provincia pero **otra descripción** se conservan: pueden ser vacantes distintas (otra sucursal, otro turno u otros requisitos).

In [298]:
filas_antes = len(df)
print("Filas idénticas:", df.duplicated().sum())

# 1. Mismo aviso cargado dos veces (solo cambian id, url y extraido)
columnas_contenido = [c for c in df.columns if c not in ["id", "url", "extraido"]]
repetidos = df.duplicated(subset=columnas_contenido)
df = df[~repetidos]
print("Avisos cargados dos veces con distinto id:", repetidos.sum())
anotar("todas menos id, url, extraido", f"Eliminados {repetidos.sum()} avisos con el mismo contenido cargados dos veces con distinto id")

# 2. Republicaciones: se ordena por fecha y se conserva la primera publicación
clave_aviso = ["titulo", "empresa", "provincia", "descripcion"]
df = df.sort_values("fecha_publicacion")
fechas = df[df.duplicated(subset=clave_aviso, keep=False)].groupby(clave_aviso, dropna=False)["fecha_publicacion"]
dias_entre = (fechas.max() - fechas.min()).dt.days
republicados = df.duplicated(subset=clave_aviso, keep="first")
df = df[~republicados].sort_values("fecha_publicacion", ascending=False).reset_index(drop=True)

print(f"Republicaciones eliminadas: {republicados.sum()} "
      f"(mediana de {dias_entre.median():.0f} días entre la primera y la última publicación)")
anotar("titulo, empresa, provincia, descripcion",
       f"Eliminadas {republicados.sum()} republicaciones de la misma vacante (mediana de {dias_entre.median():.0f} días entre publicaciones); "
       "se conserva la primera para no inflar la demanda")

# Avisos parecidos que se conservan
parecidos = df.duplicated(subset=["titulo", "empresa", "provincia"], keep=False).sum()
print(f"Avisos con igual título, empresa y provincia pero otra descripción (se conservan): {parecidos}")
anotar("titulo, empresa, provincia", f"Se conservan {parecidos} avisos con igual título, empresa y provincia pero otra descripción: pueden ser vacantes distintas")

print(f"\nFilas: {filas_antes:,} → {len(df):,} ({filas_antes - len(df)} eliminadas)")

Filas idénticas: 0
Avisos cargados dos veces con distinto id: 8
Republicaciones eliminadas: 43 (mediana de 20 días entre la primera y la última publicación)
Avisos con igual título, empresa y provincia pero otra descripción (se conservan): 162

Filas: 3,011 → 2,960 (51 eliminadas)


### 4.5 Idiomas
`idiomas` junta varios idiomas y niveles en un solo texto (p. ej. «Español (Nativo); Inglés (Básico)»). Para la pregunta 3 se extrae el **nivel de inglés** pedido en una columna propia. Un aviso sin idiomas no pide un idioma, así que se marca como «No especifica» en lugar de imputarlo.

In [299]:
NIVELES_INGLES = ["No especifica", "Básico", "Intermedio", "Avanzado", "Nativo"]

df["nivel_ingles"] = df["idiomas"].str.extract(r"Inglés \((\w+)\)", expand=False).fillna("No especifica")
df["pide_ingles"] = df["nivel_ingles"] != "No especifica"
df["idiomas"] = df["idiomas"].fillna("No especifica")

print(df["nivel_ingles"].value_counts().to_string())
anotar("nivel_ingles, pide_ingles", f"Columnas nuevas extraídas de «idiomas»: nivel de inglés pedido y si lo pide ({df['pide_ingles'].mean():.1%} de los avisos)")
anotar("idiomas", "Vacíos marcados como «No especifica»: el aviso no pide idioma; no se imputa")

nivel_ingles
No especifica    2661
Avanzado          119
Intermedio        100
Básico             73
Nativo              7


### 4.6 Atípicos
**Detección:** se calcula el rango intercuartílico (IQR) de cada variable numérica y se revisan los extremos.

**Decisión:**
- **Salarios:** el IQR no sirve como regla de corte porque la distribución es muy asimétrica; su límite superior marcaría como atípicos salarios gerenciales normales. Se usa en cambio un rango válido de salario mensual:
  - **Menos de B/. 250:** no es un salario mensual. Casi todos son valores de relleno como 1 y 2 («a convenir») o montos por hora o por día.
  - **Más de B/. 10,000:** son muy pocos y no son creíbles para el puesto (p. ej. un asistente contable con B/. 12,000 o un desarrollador junior con B/. 1,350,000). Probablemente son salarios anuales o errores de digitación.

  El valor fuera de rango se deja vacío, pero el aviso no se elimina: sigue contando para la demanda.
- **Edad:** una edad máxima mayor a 65 años equivale a no poner límite; se deja vacía.
- **Vacantes y experiencia:** sus extremos son plausibles (p. ej. 50 vacantes en un call center, 10 años de experiencia para gerencia) y se conservan.

In [300]:
def limites_iqr(serie):
    """Límites inferior y superior según la regla de 1.5 × IQR."""
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

# Detección: límites IQR, cantidad de atípicos y extremos de cada variable numérica
deteccion = []
for col in columnas_numericas:
    inferior, superior = limites_iqr(df[col].dropna())
    deteccion.append({
        "columna": col,
        "limite_inferior": inferior,
        "limite_superior": superior,
        "atipicos_iqr": int(((df[col] < inferior) | (df[col] > superior)).sum()),
        "minimo": df[col].min(),
        "maximo": df[col].max(),
    })
pd.DataFrame(deteccion).round(1)

,columna,limite_inferior,limite_superior,atipicos_iqr,minimo,maximo
0,vacantes,1.0,1.0,617,1.0,50.0
1,salario_min,75.0,1555.0,73,1.0,900000.0
2,salario_max,-425.0,2575.0,39,0.0,1350000.0
3,edad_min,9.5,37.5,7,18.0,45.0
4,edad_max,22.5,58.5,11,23.0,99.0
5,experiencia_meses,-24.0,72.0,18,12.0,120.0


In [301]:
# Avisos con salarios por encima del rango válido (se revisan uno por uno)
SALARIO_MINIMO_VALIDO = 250      # B/. mensuales
SALARIO_MAXIMO_VALIDO = 10_000   # B/. mensuales
EDAD_MAXIMA_VALIDA = 65          # años

muy_altos = (df["salario_min"] > SALARIO_MAXIMO_VALIDO) | (df["salario_max"] > SALARIO_MAXIMO_VALIDO)
df.loc[muy_altos, ["titulo", "nivel_laboral", "salario_min", "salario_max"]]

,titulo,nivel_laboral,salario_min,salario_max
727,Director(a) de Logística Integral – México y Panamá,Senior,19900.0,20000.0
1233,Agente Comercial (Panama Metro),Junior,NaN,72176.0
1721,Asesor de ventas,Otro,NaN,62816.0
1879,Asistente Contable y Administrativo,NaN,12000.0,12001.0
2033,Desarrollador Junior Full Stack (Móvil y Web con IA),Semi Sr,900000.0,1350000.0


In [302]:
salario_antes = df[["salario_min", "salario_max"]].copy()

# Salarios fuera del rango válido → vacíos (el aviso se conserva)
for col in ["salario_min", "salario_max"]:
    fuera = df[col].notna() & ~df[col].between(SALARIO_MINIMO_VALIDO, SALARIO_MAXIMO_VALIDO)
    df[col] = df[col].where(~fuera)
    print(f"{col}: {fuera.sum()} valores fuera del rango B/. {SALARIO_MINIMO_VALIDO}–{SALARIO_MAXIMO_VALIDO:,}")
    anotar(col, f"{fuera.sum()} valores fuera del rango B/. {SALARIO_MINIMO_VALIDO}–{SALARIO_MAXIMO_VALIDO:,} mensuales "
                "(valores de relleno, montos por hora/día o anuales) → vacíos; el aviso se conserva")

# Si el mínimo quedara mayor que el máximo, se intercambian
invertidos = df["salario_min"] > df["salario_max"]
df.loc[invertidos, ["salario_min", "salario_max"]] = df.loc[invertidos, ["salario_max", "salario_min"]].values
print("Salarios con mínimo mayor que el máximo:", invertidos.sum())

# Salario de referencia del aviso: promedio entre mínimo y máximo (o el único que tenga)
df["salario_medio"] = df[["salario_min", "salario_max"]].mean(axis=1)
anotar("salario_medio", "Columna nueva: promedio entre salario mínimo y máximo (o el único disponible), en B/. mensuales brutos")

# Edad máxima mayor a 65 → sin límite (vacío)
edad_alta = df["edad_max"] > EDAD_MAXIMA_VALIDA
df["edad_max"] = df["edad_max"].where(~edad_alta)
print(f"edad_max: {edad_alta.sum()} valores mayores a {EDAD_MAXIMA_VALIDA} años")
anotar("edad_max", f"{edad_alta.sum()} edades máximas mayores a {EDAD_MAXIMA_VALIDA} años equivalen a no poner límite → vacías")
anotar("vacantes, experiencia_meses", "Extremos revisados y plausibles (p. ej. 50 vacantes en call center, 10 años de experiencia); se conservan")

salario_min: 43 valores fuera del rango B/. 250–10,000
salario_max: 38 valores fuera del rango B/. 250–10,000
Salarios con mínimo mayor que el máximo: 0
edad_max: 3 valores mayores a 65 años


In [303]:
# Salario máximo antes y después de tratar los atípicos (escala logarítmica para ver los extremos)
comparacion_salario = pd.concat([
    pd.DataFrame({"salario_max": salario_antes["salario_max"], "momento": "Antes"}),
    pd.DataFrame({"salario_max": df["salario_max"], "momento": "Después"}),
]).dropna()

fig = px.box(comparacion_salario, x="momento", y="salario_max", log_y=True, points="outliers",
             title="Los salarios atípicos se concentran en valores de relleno y montos no mensuales",
             labels={"salario_max": "Salario máximo (B/. mensuales, escala log)", "momento": ""})
fig.show()

### 4.7 Faltantes

| Columnas | Decisión | Por qué |
|---|---|---|
| `empresa` | «Confidencial» | Está vacía justo en los avisos confidenciales: la empresa ocultó su nombre. Además anonimiza. |
| `industria`, `empresa_tamano`, `localidad`, `nivel_laboral`, `contratacion`, `educacion`, `educacion_estado` | «Sin dato» | Son categóricas: imputar la moda inventaría información (p. ej. haría parecer que casi todos piden «Universitario»). Marcarlas permite contarlas aparte. |
| `salario_min`, `salario_max`, `salario_medio` | Vacíos + `publica_salario` | Falta en la gran mayoría de avisos. Imputar con la media o la mediana fabricaría un salario para casi todos y ocultaría el hallazgo de que pocas empresas lo publican. El análisis salarial usa solo los avisos con salario. |
| `edad_min`, `edad_max` | Vacíos + `fija_edad` | Vacío significa que el aviso no fija edad; no es un dato perdido. |
| `experiencia_meses` | Vacío + `pide_experiencia` | Vacío significa que el aviso no especifica experiencia; imputar cambiaría los requisitos reales. |

In [304]:
faltantes_previos = df.isna().sum()

# empresa: vacía exactamente en los avisos confidenciales
coincide = (df["empresa"].isna() == df["empresa_confidencial"]).mean()
vacias = df["empresa"].isna().sum()
df["empresa"] = df["empresa"].fillna("Confidencial")
anotar("empresa", f"{vacias} vacías → «Confidencial»; coincide con empresa_confidencial en el {coincide:.0%} de los avisos (también anonimiza)")

# Categóricas: «Sin dato»
columnas_sin_dato = ["industria", "empresa_tamano", "localidad", "nivel_laboral", "contratacion", "educacion", "educacion_estado"]
df[columnas_sin_dato] = df[columnas_sin_dato].fillna("Sin dato")
anotar(", ".join(columnas_sin_dato), "Vacíos marcados como «Sin dato»: no se imputa la moda para no inventar categorías")

# Banderas: permiten medir qué tan común es publicar salario, fijar edad o pedir experiencia
df["publica_salario"] = df["salario_medio"].notna()
df["fija_edad"] = df[["edad_min", "edad_max"]].notna().any(axis=1)
df["pide_experiencia"] = df["experiencia_meses"].notna()

anotar("salario_min, salario_max, salario_medio, publica_salario",
       f"No se imputan: solo el {df['publica_salario'].mean():.1%} publica un salario válido y la media fabricaría salarios; "
       "se crea publica_salario y el análisis salarial usa solo esos avisos")
anotar("edad_min, edad_max, fija_edad", f"No se imputan: vacío = el aviso no fija edad; se crea fija_edad ({df['fija_edad'].mean():.1%} la fija)")
anotar("experiencia_meses, pide_experiencia", f"No se imputa: vacío = no especifica experiencia; se crea pide_experiencia ({df['pide_experiencia'].mean():.1%} la pide)")

# Faltantes antes y después de este paso (solo columnas que tenían alguno)
tabla_faltantes = pd.DataFrame({"antes": faltantes_previos, "despues": df.isna().sum()}).dropna()
tabla_faltantes[tabla_faltantes["antes"] > 0].astype(int)

,antes,despues
contratacion,595,0
edad_max,2312,2312
edad_min,2309,2309
educacion,1762,0
educacion_estado,1762,0
empresa,1140,0
empresa_tamano,1166,0
experiencia_meses,2120,2120
industria,1198,0
localidad,41,0


### 4.8 Columnas constantes y sensibles
- Las columnas con **un solo valor** no aportan información y se eliminan. Se detectan con `nunique()`, así que si una nueva extracción trae más valores, se conservan.
- `url` y `descripcion` se eliminan del archivo limpio para no publicar enlaces ni textos completos de los avisos (anonimización). La descripción ya se usó en 4.4 para detectar republicaciones.

In [305]:
# Columnas con un solo valor (sin contar vacíos)
columnas_nuevas = ["nivel_ingles", "pide_ingles", "salario_medio", "publica_salario", "fija_edad", "pide_experiencia"]   # se conservan siempre
constantes = [c for c in df.columns if df[c].nunique() <= 1 and c not in columnas_nuevas]
for col in constantes:
    valor = df[col].dropna().unique()
    anotar(col, f"Eliminada: un solo valor en toda la columna («{valor[0] if len(valor) else 'vacía'}»), no aporta información")

columnas_sensibles = ["url", "descripcion"]
df = df.drop(columns=constantes + columnas_sensibles)
anotar(", ".join(columnas_sensibles), "Eliminadas del archivo limpio por anonimización: no se publican enlaces ni textos completos")

print("Columnas constantes eliminadas:", constantes)
print("Columnas sensibles eliminadas:", columnas_sensibles)

Columnas constantes eliminadas: ['salario_frecuencia', 'salario_tipo']
Columnas sensibles eliminadas: ['url', 'descripcion']


### 4.9 Tipo categoría
Las columnas de texto con pocos valores que se repiten pasan a `category`. Las **ordinales** se ordenan de menor a mayor, con «Otro» y «Sin dato» al final, fuera de la escala.

Si una nueva extracción trae un valor que no está en el orden, se agrega antes de «Otro» / «Sin dato» y se muestra un aviso, para no perder datos. El diccionario `ORDENES` se reutiliza al leer el CSV limpio, porque el CSV no guarda el tipo categoría.

In [306]:
ORDEN_NIVEL = ["Sin Experiencia", "Trainee / Pasante", "Junior", "Semi Sr", "Senior",
               "Jefe / Supervisor / Responsable", "Gerencia / Alta Gerencia / Dirección", "Otro", "Sin dato"]
ORDEN_EDUCACION = ["Secundario", "Terciario/Técnico", "Universitario", "Posgrado", "Maestría", "Doctorado", "Otro", "Sin dato"]

def primer_numero(texto):
    """Primer número de un texto («Entre 11 y 50» → 11); infinito si no tiene números."""
    encontrado = re.search(r"\d+", texto)
    return int(encontrado.group()) if encontrado else float("inf")

# Tamaño de empresa: se ordena por el primer número del rango
ORDEN_TAMANO = sorted([t for t in df["empresa_tamano"].unique() if t != "Sin dato"], key=primer_numero) + ["Sin dato"]

ORDENES = {
    "nivel_laboral": ORDEN_NIVEL,
    "educacion": ORDEN_EDUCACION,
    "empresa_tamano": ORDEN_TAMANO,
    "nivel_ingles": NIVELES_INGLES,
}

def ordenar(serie, orden):
    """Convierte a categoría ordenada; los valores nuevos se agregan antes de «Otro» / «Sin dato»."""
    nuevos = [v for v in serie.dropna().unique() if v not in orden]
    if nuevos:
        print(f"Aviso: valores nuevos en {serie.name}: {nuevos}")
        al_final = [v for v in orden if v in ("Otro", "Sin dato")]
        orden = [v for v in orden if v not in al_final] + nuevos + al_final
    return pd.Categorical(serie, categories=orden, ordered=True)

for col, orden in ORDENES.items():
    df[col] = ordenar(df[col], orden)

columnas_categoricas = ["empresa", "industria", "provincia", "localidad", "area", "subarea",
                        "tipo_trabajo", "modalidad", "contratacion", "educacion_estado"]
for col in columnas_categoricas:
    df[col] = df[col].astype("category")

anotar(", ".join(ORDENES), "Convertidas a categoría ordenada para respetar su orden natural en tablas y gráficos")
anotar(", ".join(columnas_categoricas), "Convertidas a tipo category: pocos valores que se repiten")

# Orden aplicado a cada columna ordinal
pd.Series({col: " < ".join(df[col].cat.categories) for col in ORDENES}, name="orden")

nivel_laboral     Sin Experiencia < Trainee / Pasante < Junior < Semi Sr < Senior < Jefe / Sup...
educacion         Secundario < Terciario/Técnico < Universitario < Posgrado < Maestría < Docto...
empresa_tamano    Entre 1 y 10 < Entre 11 y 50 < Entre 51 y 150 < Entre 151 y 300 < Entre 301 ...
nivel_ingles                              No especifica < Básico < Intermedio < Avanzado < Nativo
Name: orden, dtype: str

### 4.10 Bitácora de limpieza
Resumen de todas las transformaciones hechas sobre los datos originales y su justificación.

In [307]:
bitacora = pd.DataFrame(registros, columns=["Columna", "Justificación"])

with pd.option_context("display.max_colwidth", None):
    display(bitacora)

,Columna,Justificación
0,todas,"Nombres verificados en snake_case (minúsculas, sin tildes ni espacios); columnas renombradas: 0"
1,"fecha_publicacion, fecha_finalizacion, extraido",Convertidas a datetime para analizar la evolución en el tiempo («extraido» venía como texto)
2,"vacantes, salario_min, salario_max, edad_min, edad_max, experiencia_meses",Convertidas a número (errors='coerce') para calcular estadísticos
3,"empresa_confidencial, apto_discapacidad",Aseguradas como booleanas (True/False)
4,"empresa, localidad, subarea, descripcion","Quitados espacios al inicio/final y espacios dobles (3,201 valores)"
5,titulo,748 títulos en MAYÚSCULAS pasados a formato oración para uniformar el texto
6,localidad,"Unificadas variantes del mismo lugar (idioma, tildes, puntos, «District»), p. ej. «Panama City» → «Ciudad de Panamá», «Betania» → «Bethania» (1375 valores)"
7,area,"Agregadas las tildes faltantes («Tecnico», «Maritimo») (36 valores)"
8,industria,«Financiera» y «Banca / Financiera» son el mismo sector (107 valores)
9,nivel_laboral,"«Senior / Semi-Senior» acepta ambos niveles: se asigna al menor, «Semi Sr». Experiencia mediana pedida: 24 meses, igual que Semi Sr (24) (193 valores)"


## 5. Perfil después de la limpieza
Se repite el perfil de la sección 3 para comparar el antes y el después, y se guarda el archivo limpio en `datos/limpios/`.

In [308]:
perfil_despues = perfil(df)
resumen_despues = resumen(df)

# Resumen general antes y después
pd.DataFrame({"antes": resumen_antes, "despues": resumen_despues})

,antes,despues
filas,3011,2960
columnas,31,33
celdas_vacias,27627,14320
pct_celdas_vacias,29.6,14.7
columnas_con_faltantes,16,6
filas_duplicadas,0,0


In [309]:
# Tipo y % de faltantes por columna, antes y después
orden_columnas = list(crudos.columns) + [c for c in df.columns if c not in crudos.columns]
comparacion_columnas = pd.DataFrame({
    "tipo_antes": perfil_antes["tipo"],
    "tipo_despues": perfil_despues["tipo"],
    "pct_faltantes_antes": perfil_antes["pct_faltantes"],
    "pct_faltantes_despues": perfil_despues["pct_faltantes"],
}).reindex(orden_columnas)

comparacion_columnas["estado"] = "conservada"
comparacion_columnas.loc[~comparacion_columnas.index.isin(df.columns), "estado"] = "eliminada"
comparacion_columnas.loc[~comparacion_columnas.index.isin(crudos.columns), "estado"] = "nueva"
comparacion_columnas

,tipo_antes,tipo_despues,pct_faltantes_antes,pct_faltantes_despues,estado
id,int64,int64,0.0,0.0,conservada
titulo,str,str,0.0,0.0,conservada
empresa,str,category,38.7,0.0,conservada
empresa_confidencial,bool,bool,0.0,0.0,conservada
industria,str,category,40.6,0.0,conservada
empresa_tamano,str,category,39.6,0.0,conservada
fecha_publicacion,datetime64[us],datetime64[us],0.0,0.0,conservada
fecha_finalizacion,datetime64[us],datetime64[us],0.0,0.0,conservada
provincia,str,category,0.0,0.0,conservada
localidad,str,category,1.4,0.0,conservada


In [310]:
# Guardar el dataset limpio (utf-8-sig para que Excel muestre bien las tildes)
DIR_LIMPIOS.mkdir(parents=True, exist_ok=True)
ARCHIVO_LIMPIO = DIR_LIMPIOS / "konzerta_limpio.csv"
df.to_csv(ARCHIVO_LIMPIO, index=False, encoding="utf-8-sig")

print(f"Guardado: {ARCHIVO_LIMPIO.relative_to(BASE)} ({df.shape[0]:,} filas × {df.shape[1]} columnas)")

Guardado: datos/limpios/konzerta_limpio.csv (2,960 filas × 33 columnas)


## 6. Tablas de análisis
Tablas resumidas que responden las preguntas de la propuesta. Cada una alimenta uno o más gráficos del EDA y del dashboard, y se guarda en `datos/limpios/`.

| Tabla | Pregunta | Gráfico del dashboard |
|---|---|---|
| `t1_demanda_area` | P1 · ¿Qué áreas concentran la demanda? | Avisos por área |
| `t2_demanda_provincia` | P1 · ¿Qué provincias concentran la demanda? | Avisos por provincia |
| `t3_demanda_industria` | P1 · ¿Qué industrias concentran la demanda? | Avisos por industria |
| `t4_salario_area` | P2 · ¿Cuánto se paga por área y qué tan común es publicarlo? | Salario por área |
| `t5_salario_nivel` | P2 · ¿Cuánto se paga por nivel y qué tan común es publicarlo? | (EDA) |
| `t6_requisitos_nivel` | P3 · ¿Qué requisitos se piden por nivel? | Requisitos por nivel |
| `t7_semanal` | P4 · ¿Cómo evoluciona la publicación semanal? | Avisos por semana |
| `t8_modalidad` | P4 · ¿Qué peso tienen el trabajo remoto e híbrido? | Indicadores de modalidad |
| `t9_modalidad_area` | P4 · ¿En qué áreas hay más trabajo híbrido o remoto? | Híbrido o remoto por área |
| `t10_perfil_area` | P1 a P4 · ¿Cómo es cada área frente al mercado? | Explorador «Tu área» |

Los porcentajes van de 0 a 100 y los salarios están en B/. mensuales brutos.

### 6.1 Lectura del dataset limpio
Las tablas se calculan desde el archivo guardado en `datos/limpios/`, que es lo mismo que leerá el dashboard. El CSV no guarda las fechas ni el tipo categoría, así que `leer_limpio()` los recupera con `ORDENES` (sección 4.9).

In [311]:
def leer_limpio(ruta=ARCHIVO_LIMPIO):
    """Lee el CSV limpio y recupera los tipos que el CSV no guarda (fechas y categorías)."""
    datos = pd.read_csv(ruta, parse_dates=["fecha_publicacion", "fecha_finalizacion", "extraido"])
    for col, orden in ORDENES.items():
        datos[col] = ordenar(datos[col], orden)
    for col in columnas_categoricas:
        datos[col] = datos[col].astype("category")
    return datos

limpio = leer_limpio()

# Comprobación: el CSV leído tiene las mismas filas, columnas y tipos que el DataFrame limpio
tipos_distintos = [c for c in df.columns if str(limpio[c].dtype).split("[")[0] != str(df[c].dtype).split("[")[0]]
print(f"Leído: {limpio.shape[0]:,} filas × {limpio.shape[1]} columnas")
print("Columnas con un tipo distinto al de df:", tipos_distintos or "ninguna")

Leído: 2,960 filas × 33 columnas
Columnas con un tipo distinto al de df: ninguna


In [312]:
MIN_AVISOS_SALARIO = 10   # mínimo de avisos con salario para comparar un grupo

def demanda(datos, grupo):
    """Avisos y vacantes por grupo, con su porcentaje y el porcentaje acumulado de avisos."""
    tabla = datos.groupby(grupo, observed=True).agg(avisos=("id", "size"), vacantes=("vacantes", "sum"))
    tabla = tabla.sort_values("avisos", ascending=False)
    tabla["pct_avisos"] = tabla["avisos"] / tabla["avisos"].sum() * 100
    tabla["pct_vacantes"] = tabla["vacantes"] / tabla["vacantes"].sum() * 100
    tabla["pct_acumulado"] = tabla["pct_avisos"].cumsum()   # se acumula antes de redondear
    return tabla.round(1).reset_index()

def resumen_salario(datos, grupo):
    """Por grupo: avisos, avisos con salario, % que lo publica y cuartiles del salario medio."""
    grupos = datos.groupby(grupo, observed=True)
    tabla = pd.DataFrame({"avisos": grupos.size(), "con_salario": grupos["publica_salario"].sum()})
    tabla["pct_publica_salario"] = (tabla["con_salario"] / tabla["avisos"] * 100).round(1)

    cuartiles = (datos[datos["publica_salario"]].groupby(grupo, observed=True)["salario_medio"]
                 .quantile([0.25, 0.5, 0.75]).unstack())
    cuartiles.columns = ["salario_q1", "salario_mediana", "salario_q3"]
    tabla = tabla.join(cuartiles).round(1)

    # Un grupo con muy pocos salarios no se compara: su mediana no es representativa
    tabla["n_suficiente"] = tabla["con_salario"] >= MIN_AVISOS_SALARIO
    return tabla.reset_index()

### 6.2 Pregunta 1 · ¿Dónde se concentra la demanda?
`t1` y `t2` incluyen el **porcentaje acumulado**, que muestra cuántas áreas o provincias reúnen la mayor parte de los avisos.

In [313]:
# t1: demanda por área
t1_demanda_area = demanda(limpio, "area")
t1_demanda_area

,area,avisos,vacantes,pct_avisos,pct_vacantes,pct_acumulado
0,"Comercial, Ventas y Negocios",783,2713,26.5,36.3,26.5
1,"Administración, Contabilidad y Finanzas",442,621,14.9,8.3,41.4
2,Abastecimiento y Logística,290,615,9.8,8.2,51.2
3,Oficios y Otros,223,605,7.5,8.1,58.7
4,"Tecnología, Sistemas y Telecomunicaciones",144,216,4.9,2.9,63.6
5,Gastronomía y Turismo,142,313,4.8,4.2,68.4
6,Producción y Manufactura,141,267,4.8,3.6,73.1
7,"Atención al Cliente, Call Center y Telemarketing",137,1158,4.6,15.5,77.8
8,Ingenierías,116,164,3.9,2.2,81.7
9,Ingeniería Civil y Construcción,97,161,3.3,2.2,85.0


In [314]:
# t2: demanda por provincia
t2_demanda_provincia = demanda(limpio, "provincia")
t2_demanda_provincia

,provincia,avisos,vacantes,pct_avisos,pct_vacantes,pct_acumulado
0,Panamá,2312,5955,78.1,79.6,78.1
1,Colón,148,334,5.0,4.5,83.1
2,Chiriquí,142,301,4.8,4.0,87.9
3,Panamá Oeste,121,403,4.1,5.4,92.0
4,Coclé,63,145,2.1,1.9,94.1
5,Veraguas,55,151,1.9,2.0,96.0
6,Herrera,47,93,1.6,1.2,97.6
7,Otra,41,55,1.4,0.7,99.0
8,Bocas del Toro,18,21,0.6,0.3,99.6
9,Los Santos,11,15,0.4,0.2,99.9


`t3` usa solo los avisos con industria conocida: la industria falta en gran parte de los avisos (sobre todo en los confidenciales), y contarla como una categoría más ocultaría el reparto real.

In [315]:
# t3: demanda por industria (solo avisos con industria conocida)
con_industria = limpio[limpio["industria"] != "Sin dato"]
t3_demanda_industria = demanda(con_industria, "industria")

print(f"Avisos con industria conocida: {len(con_industria):,} de {len(limpio):,} ({len(con_industria) / len(limpio):.1%})")
t3_demanda_industria.head(15)

Avisos con industria conocida: 1,762 de 2,960 (59.5%)


,industria,avisos,vacantes,pct_avisos,pct_vacantes,pct_acumulado
0,Banca / Financiera,231,385,13.1,7.7,13.1
1,Construcción,111,113,6.3,2.3,19.4
2,Retail,108,465,6.1,9.3,25.5
3,Consultoría,107,220,6.1,4.4,31.6
4,Comercio,104,265,5.9,5.3,37.5
5,Consultora de Recursos Humanos,103,382,5.8,7.6,43.4
6,Alimenticia,94,222,5.3,4.4,48.7
7,Otra,88,182,5.0,3.6,53.7
8,Servicios,86,657,4.9,13.2,58.6
9,Telecomunicaciones,57,466,3.2,9.3,61.8


### 6.3 Pregunta 2 · ¿Cuánto se paga y qué tan común es publicarlo?
Solo se usan los avisos con un salario válido (`publica_salario`). La columna `n_suficiente` marca los grupos con al menos `MIN_AVISOS_SALARIO` salarios; los demás aparecen en la tabla, pero no se comparan en los gráficos.

In [316]:
# t4: salario por área, ordenado por la cantidad de avisos con salario
t4_salario_area = resumen_salario(limpio, "area").sort_values("con_salario", ascending=False).reset_index(drop=True)
t4_salario_area

,area,avisos,con_salario,pct_publica_salario,salario_q1,salario_mediana,salario_q3,n_suficiente
0,"Comercial, Ventas y Negocios",783,90,11.5,680.5,800.0,1087.5,True
1,"Administración, Contabilidad y Finanzas",442,71,16.1,800.0,1100.0,1525.0,True
2,Producción y Manufactura,141,29,20.6,667.5,800.5,950.0,True
3,Gastronomía y Turismo,142,26,18.3,647.0,699.8,837.5,True
4,Abastecimiento y Logística,290,26,9.0,678.1,750.0,918.8,True
5,"Tecnología, Sistemas y Telecomunicaciones",144,26,18.1,762.5,1492.0,3250.0,True
6,Oficios y Otros,223,26,11.7,675.0,762.5,1018.8,True
7,"Atención al Cliente, Call Center y Telemarketing",137,24,17.5,635.0,787.5,1012.5,True
8,Secretarias y Recepción,38,19,50.0,712.5,840.0,1000.0,True
9,Ingenierías,116,17,14.7,1050.0,1300.0,1400.0,True


In [317]:
# t5: salario por nivel laboral (en el orden de la escala)
t5_salario_nivel = resumen_salario(limpio, "nivel_laboral")

# ¿En qué niveles se concentran los avisos que publican salario?
reparto = t5_salario_nivel.set_index(t5_salario_nivel["nivel_laboral"].astype(str))["con_salario"] / t5_salario_nivel["con_salario"].sum()
pct_semi_o_sin_nivel = reparto.reindex(["Semi Sr", "Sin dato"], fill_value=0).sum()
print("Reparto de los avisos con salario por nivel:")
print((reparto * 100).round(1).sort_values(ascending=False).head(3).to_string())
t5_salario_nivel

Reparto de los avisos con salario por nivel:
nivel_laboral
Semi Sr     49.3
Sin dato    45.0
Junior       3.6


,nivel_laboral,avisos,con_salario,pct_publica_salario,salario_q1,salario_mediana,salario_q3,n_suficiente
0,Sin Experiencia,13,1,7.7,650.0,650.0,650.0,False
1,Trainee / Pasante,20,0,0.0,NaN,NaN,NaN,False
2,Junior,1021,16,1.6,800.0,875.0,1200.0,True
3,Semi Sr,1001,218,21.8,700.0,900.0,1231.2,True
4,Senior,298,3,1.0,843.8,887.5,1818.8,False
5,Jefe / Supervisor / Responsable,170,1,0.6,1375.0,1375.0,1375.0,False
6,Gerencia / Alta Gerencia / Dirección,81,0,0.0,NaN,NaN,NaN,False
7,Otro,104,4,3.8,681.2,725.0,837.5,False
8,Sin dato,252,199,79.0,699.8,850.0,1300.0,True


### 6.4 Pregunta 3 · ¿Qué requisitos se piden por nivel?
Para cada nivel: cuántos avisos piden experiencia (y cuánta), inglés (y de nivel avanzado), un rango de edad y estudios universitarios o superiores. Los porcentajes se calculan sobre **todos** los avisos del nivel; las medianas, solo sobre los que piden el requisito.

In [318]:
NIVELES_UNIVERSITARIOS = ["Universitario", "Posgrado", "Maestría", "Doctorado"]

# Columnas auxiliares solo para esta tabla (no modifican limpio)
aux = limpio.assign(
    ingles_avanzado=limpio["nivel_ingles"].isin(["Avanzado", "Nativo"]),
    universitario_o_mas=limpio["educacion"].isin(NIVELES_UNIVERSITARIOS),
)
grupos = aux.groupby("nivel_laboral", observed=True)

t6_requisitos_nivel = pd.DataFrame({
    "avisos": grupos.size(),
    "pct_pide_experiencia": grupos["pide_experiencia"].mean() * 100,
    "experiencia_mediana_meses": grupos["experiencia_meses"].median(),
    "pct_pide_ingles": grupos["pide_ingles"].mean() * 100,
    "pct_ingles_avanzado": grupos["ingles_avanzado"].mean() * 100,
    "pct_fija_edad": grupos["fija_edad"].mean() * 100,
    "edad_min_mediana": grupos["edad_min"].median(),
    "edad_max_mediana": grupos["edad_max"].median(),
    "pct_universitario_o_mas": grupos["universitario_o_mas"].mean() * 100,
}).round(1).reset_index()
t6_requisitos_nivel

,nivel_laboral,avisos,pct_pide_experiencia,experiencia_mediana_meses,pct_pide_ingles,pct_ingles_avanzado,pct_fija_edad,edad_min_mediana,edad_max_mediana,pct_universitario_o_mas
0,Sin Experiencia,13,23.1,12.0,0.0,0.0,53.8,22.0,45.0,0.0
1,Trainee / Pasante,20,5.0,12.0,10.0,0.0,15.0,20.0,35.0,40.0
2,Junior,1021,28.5,12.0,8.9,3.1,22.6,23.0,40.0,21.4
3,Semi Sr,1001,24.9,24.0,9.1,3.7,18.1,25.0,40.0,27.8
4,Senior,298,43.0,24.0,14.4,5.7,33.9,25.0,45.0,34.2
5,Jefe / Supervisor / Responsable,170,49.4,36.0,14.7,6.5,34.7,28.0,45.0,47.6
6,Gerencia / Alta Gerencia / Dirección,81,38.3,60.0,27.2,14.8,32.1,30.0,45.0,60.5
7,Otro,104,39.4,12.0,21.2,15.4,32.7,20.0,45.0,12.5
8,Sin dato,252,4.8,24.0,1.2,0.4,3.6,25.0,45.0,2.8


### 6.5 Pregunta 4 · Evolución semanal y modalidad
**Cuidado con las semanas antiguas.** El dataset es una foto de los avisos **vigentes** el día de la extracción. Un aviso dura un tiempo limitado, así que los publicados mucho antes de la extracción ya habían vencido y no aparecen. Por eso las primeras semanas muestran muchos menos avisos sin que eso signifique que hubo menos demanda.

La columna `semana_comparable` marca las semanas que se pueden comparar entre sí: empiezan después de *fecha de extracción − duración típica de un aviso* (la mediana de `fecha_finalizacion − fecha_publicacion`) y terminan antes de la extracción (la última semana está incompleta).

In [319]:
# Duración típica de un aviso y primer día en que todos los avisos publicados siguen vigentes
duracion_tipica = (limpio["fecha_finalizacion"] - limpio["fecha_publicacion"]).dt.days.median()
fecha_extraccion = limpio["extraido"].max().normalize()
inicio_comparable = fecha_extraccion - pd.Timedelta(days=duracion_tipica)

# Semana de publicación (lunes de cada semana)
semana = limpio["fecha_publicacion"].dt.to_period("W-SUN").dt.start_time.rename("semana")

t7_semanal = pd.crosstab(semana, limpio["modalidad"])
t7_semanal.columns = t7_semanal.columns.astype(str)
t7_semanal.columns.name = None
t7_semanal.insert(0, "avisos", t7_semanal.sum(axis=1))
t7_semanal.insert(1, "vacantes", limpio.groupby(semana)["vacantes"].sum())
t7_semanal["semana_comparable"] = ((t7_semanal.index >= inicio_comparable) &
                                   (t7_semanal.index + pd.Timedelta(days=6) <= fecha_extraccion))
t7_semanal = t7_semanal.reset_index()

print(f"Duración típica de un aviso: {duracion_tipica:.0f} días · Extracción: {fecha_extraccion:%d/%m/%Y}")
print(f"Semanas comparables: desde el {inicio_comparable:%d/%m/%Y} "
      f"({t7_semanal['semana_comparable'].sum()} de {len(t7_semanal)} semanas)")
t7_semanal

Duración típica de un aviso: 60 días · Extracción: 08/10/2026
Semanas comparables: desde el 09/08/2026 (8 de 14 semanas)


,semana,avisos,vacantes,Híbrido,Presencial,Remoto,semana_comparable
0,2026-07-06,2,2,0,2,0,False
1,2026-07-13,40,77,0,40,0,False
2,2026-07-20,32,86,0,32,0,False
3,2026-07-27,30,53,0,30,0,False
4,2026-08-03,35,63,0,35,0,False
5,2026-08-10,274,648,7,265,2,True
6,2026-08-17,294,542,10,283,1,True
7,2026-08-24,279,595,5,273,1,True
8,2026-08-31,336,622,2,334,0,True
9,2026-09-07,404,1063,7,395,2,True


In [320]:
# t8: modalidad de trabajo
t8_modalidad = demanda(limpio, "modalidad")
t8_modalidad

,modalidad,avisos,vacantes,pct_avisos,pct_vacantes,pct_acumulado
0,Presencial,2893,7311,97.7,97.8,97.7
1,Híbrido,57,89,1.9,1.2,99.7
2,Remoto,10,77,0.3,1.0,100.0


In [321]:
# t9: avisos híbridos o remotos por área (solo áreas con suficientes avisos)
MIN_AVISOS_AREA = 30

t9_modalidad_area = (limpio.assign(no_presencial=limpio["modalidad"] != "Presencial")
                     .groupby("area", observed=True)
                     .agg(avisos=("id", "size"), hibrido_o_remoto=("no_presencial", "sum")))
t9_modalidad_area["pct_hibrido_o_remoto"] = (t9_modalidad_area["hibrido_o_remoto"] / t9_modalidad_area["avisos"] * 100).round(1)
t9_modalidad_area = (t9_modalidad_area[t9_modalidad_area["avisos"] >= MIN_AVISOS_AREA]
                     .sort_values("pct_hibrido_o_remoto", ascending=False).reset_index())
t9_modalidad_area

,area,avisos,hibrido_o_remoto,pct_hibrido_o_remoto
0,Marketing y Publicidad,95,10,10.5
1,"Atención al Cliente, Call Center y Telemarketing",137,10,7.3
2,"Tecnología, Sistemas y Telecomunicaciones",144,10,6.9
3,Recursos Humanos y Capacitación,69,3,4.3
4,"Administración, Contabilidad y Finanzas",442,17,3.8
5,"Salud, Medicina y Farmacia",54,1,1.9
6,"Comercial, Ventas y Negocios",783,12,1.5
7,Ingenierías,116,1,0.9
8,Abastecimiento y Logística,290,2,0.7
9,Ingeniería Civil y Construcción,97,0,0.0


In [322]:
# t10: perfil de cada área, para el explorador del dashboard
grupos_area = aux.assign(no_presencial=aux["modalidad"] != "Presencial").groupby("area", observed=True)

t10_perfil_area = pd.DataFrame({
    "avisos": grupos_area.size(),
    "vacantes": grupos_area["vacantes"].sum(),
    "pct_publica_salario": grupos_area["publica_salario"].mean() * 100,
    "pct_pide_experiencia": grupos_area["pide_experiencia"].mean() * 100,
    "pct_pide_ingles": grupos_area["pide_ingles"].mean() * 100,
    "pct_universitario_o_mas": grupos_area["universitario_o_mas"].mean() * 100,
    "pct_hibrido_o_remoto": grupos_area["no_presencial"].mean() * 100,
}).round(1).sort_values("avisos", ascending=False).reset_index()

# Solo se muestra la mediana salarial de las áreas con suficientes salarios
salario_comparable = t4_salario_area[t4_salario_area["n_suficiente"]].set_index("area")["salario_mediana"]
t10_perfil_area["salario_mediana"] = t10_perfil_area["area"].map(salario_comparable).astype(float)
t10_perfil_area.head()

,area,avisos,vacantes,pct_publica_salario,pct_pide_experiencia,pct_pide_ingles,pct_universitario_o_mas,pct_hibrido_o_remoto,salario_mediana
0,"Comercial, Ventas y Negocios",783,2713,11.5,27.8,9.1,22.2,1.5,800.0
1,"Administración, Contabilidad y Finanzas",442,621,16.1,25.6,10.4,35.3,3.8,1100.0
2,Abastecimiento y Logística,290,615,9.0,34.8,6.9,23.1,0.7,750.0
3,Oficios y Otros,223,605,11.7,27.4,2.7,4.9,0.0,762.5
4,"Tecnología, Sistemas y Telecomunicaciones",144,216,18.1,22.9,16.7,46.5,6.9,1492.0


### 6.6 Guardado de las tablas
Cada tabla se guarda como CSV en `datos/limpios/`. El diccionario `tablas` las reúne para el EDA y el dashboard.

In [323]:
tablas = {
    "t1_demanda_area": t1_demanda_area,
    "t2_demanda_provincia": t2_demanda_provincia,
    "t3_demanda_industria": t3_demanda_industria,
    "t4_salario_area": t4_salario_area,
    "t5_salario_nivel": t5_salario_nivel,
    "t6_requisitos_nivel": t6_requisitos_nivel,
    "t7_semanal": t7_semanal,
    "t8_modalidad": t8_modalidad,
    "t9_modalidad_area": t9_modalidad_area,
    "t10_perfil_area": t10_perfil_area,
}

for nombre, tabla in tablas.items():
    tabla.to_csv(DIR_LIMPIOS / f"{nombre}.csv", index=False, encoding="utf-8-sig")
    print(f"Guardada: datos/limpios/{nombre}.csv ({tabla.shape[0]} filas × {tabla.shape[1]} columnas)")

Guardada: datos/limpios/t1_demanda_area.csv (25 filas × 6 columnas)
Guardada: datos/limpios/t2_demanda_provincia.csv (11 filas × 6 columnas)
Guardada: datos/limpios/t3_demanda_industria.csv (53 filas × 6 columnas)
Guardada: datos/limpios/t4_salario_area.csv (25 filas × 8 columnas)
Guardada: datos/limpios/t5_salario_nivel.csv (9 filas × 8 columnas)
Guardada: datos/limpios/t6_requisitos_nivel.csv (9 filas × 10 columnas)
Guardada: datos/limpios/t7_semanal.csv (14 filas × 7 columnas)
Guardada: datos/limpios/t8_modalidad.csv (3 filas × 6 columnas)
Guardada: datos/limpios/t9_modalidad_area.csv (16 filas × 4 columnas)
Guardada: datos/limpios/t10_perfil_area.csv (25 filas × 9 columnas)


## 7. Análisis exploratorio (EDA)
Se trabaja sobre `limpio` (el CSV limpio) y las tablas de la sección 6. Contenido:

1. Perfil antes y después
2. Estadísticos descriptivos de las variables numéricas
3. Frecuencias de las variables categóricas
4. Distribuciones
5. Comparaciones entre grupos
6. Relaciones entre variables
7. Tendencia en el tiempo
8. Hallazgos
9. Contraste con lo esperado en la propuesta

Cada gráfico lleva un título con su hallazgo, ejes con unidades y una interpretación. Las cifras de los títulos y de las interpretaciones se calculan desde los datos.

Cada interpretación guarda su gráfico y sus tablas en `eda`; con eso se arma la pestaña «Análisis exploratorio» del dashboard (ver 8.5).

In [324]:
import numpy as np
from IPython.display import Markdown

px.defaults.template = "plotly_white"

ESCALA_NIVEL = [n for n in ORDEN_NIVEL if n not in ("Otro", "Sin dato")]   # niveles con orden jerárquico
MESES = {1: "enero", 2: "febrero", 3: "marzo", 4: "abril", 5: "mayo", 6: "junio", 7: "julio",
         8: "agosto", 9: "septiembre", 10: "octubre", 11: "noviembre", 12: "diciembre"}
DIAS = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]

con_salario = limpio[limpio["publica_salario"]]   # avisos con salario válido

eda = {}   # piezas del EDA para la pestaña del dashboard; la clave evita repetirlas si una celda se ejecuta de nuevo

def interpretar(texto, seccion, fig=None, tablas=()):
    """Muestra una línea de interpretación y guarda la pieza para la pestaña del dashboard."""
    display(Markdown(f"**Interpretación:** {texto}"))
    eda[(seccion, texto)] = {"seccion": seccion, "texto": texto, "fig": fig, "tablas": tablas}

### 7.1 Perfil antes y después
Resumen del perfil de las secciones 3 y 5: el dataset limpio tiene menos filas (duplicados eliminados), más columnas (variables nuevas) y muchas menos celdas vacías.

In [325]:
perfil_limpio = perfil(limpio)

comparacion = pd.DataFrame({"antes": resumen_antes, "despues": resumen(limpio)}).rename_axis("medida")
tipos = (pd.DataFrame({"antes": perfil_antes["tipo"].value_counts(), "despues": perfil_limpio["tipo"].value_counts()})
         .fillna(0).astype(int).rename_axis("tipo"))
display(comparacion)
display(tipos)

interpretar(f"Se pasó de {resumen_antes['filas']:,} a {len(limpio):,} avisos y de {resumen_antes['pct_celdas_vacias']}% "
            f"a {resumen(limpio)['pct_celdas_vacias']}% de celdas vacías. Las que quedan están en salario, edad y "
            "experiencia, donde vacío significa que el aviso no publica o no exige ese dato.", "7.1", tablas=[comparacion, tipos])

,antes,despues
medida,,
filas,3011,2960
columnas,31,33
celdas_vacias,27627,14320
pct_celdas_vacias,29.6,14.7
columnas_con_faltantes,16,6
filas_duplicadas,0,0


,antes,despues
tipo,,
bool,2,6
category,0,14
datetime64[us],2,3
float64,5,6
int64,2,2
str,20,2


**Interpretación:** Se pasó de 3,011 a 2,960 avisos y de 29.6% a 14.7% de celdas vacías. Las que quedan están en salario, edad y experiencia, donde vacío significa que el aviso no publica o no exige ese dato.

### 7.2 Estadísticos descriptivos de las variables numéricas
La **asimetría** indica hacia qué lado se estira la distribución: un valor mayor a 1 significa una cola larga de valores altos. En ese caso la mediana representa mejor el valor típico que la media.

In [326]:
numericas = ["vacantes", "salario_medio", "experiencia_meses", "edad_min", "edad_max"]

descriptivos = limpio[numericas].describe().T
descriptivos["asimetria"] = limpio[numericas].skew()
descriptivos = descriptivos.round(1).rename_axis("variable").rename(columns={
    "count": "avisos con dato", "mean": "media", "std": "desv. estándar", "min": "mínimo",
    "25%": "Q1", "50%": "mediana", "75%": "Q3", "max": "máximo"})   # count = avisos con la variable, no la suma
display(descriptivos)

# Avisos con dato en cada variable numérica: cuántos datos respaldan cada estadístico
con_dato = descriptivos["avisos con dato"].sort_values()
fig = px.bar(x=con_dato.values, y=con_dato.index, orientation="h", text=con_dato.values,
             title=f"El salario solo aparece en {con_dato['salario_medio']:,.0f} de {len(limpio):,} avisos "
                   f"({con_dato['salario_medio'] / len(limpio):.0%})",
             labels={"x": "N.º de avisos con dato", "y": ""})
fig.update_traces(texttemplate="%{text:,.0f}")
fig.show()

salario = limpio["salario_medio"]
interpretar(f"El salario de los {salario.count()} avisos que lo publican tiene mediana de B/. {salario.median():,.0f} y media de "
            f"B/. {salario.mean():,.0f}: la media es mayor porque unos pocos salarios altos estiran la distribución "
            f"(asimetría {salario.skew():.1f}), así que se usa la **mediana**. En vacantes, la columna «avisos con dato» cuenta avisos, no plazas: los {len(limpio):,} avisos suman {limpio['vacantes'].sum():,.0f} vacantes "
            f"({limpio['vacantes'].mean():.1f} por aviso). El "
            f"{(limpio['vacantes'] == 1).mean():.0%} de los avisos ofrece una sola plaza, pero hay avisos de hasta "
            f"{limpio['vacantes'].max():.0f}.", "7.2", fig, tablas=[descriptivos])

,avisos con dato,media,desv. estándar,mínimo,Q1,mediana,Q3,máximo,asimetria
variable,,,,,,,,,
vacantes,2960.0,2.5,6.2,1.0,1.0,1.0,1.0,50.0,6.3
salario_medio,442.0,1098.2,654.8,300.0,700.0,875.0,1250.0,5000.0,2.3
experiencia_meses,840.0,28.4,18.6,12.0,12.0,24.0,36.0,120.0,2.0
edad_min,651.0,24.8,4.6,18.0,20.0,25.0,27.0,45.0,0.9
edad_max,648.0,42.0,6.8,23.0,36.0,43.0,45.0,65.0,0.0


**Interpretación:** El salario de los 442 avisos que lo publican tiene mediana de B/. 875 y media de B/. 1,098: la media es mayor porque unos pocos salarios altos estiran la distribución (asimetría 2.3), así que se usa la **mediana**. En vacantes, la columna «avisos con dato» cuenta avisos, no plazas: los 2,960 avisos suman 7,477 vacantes (2.5 por aviso). El 79% de los avisos ofrece una sola plaza, pero hay avisos de hasta 50.

### 7.3 Frecuencias de las variables categóricas
Los 5 valores más frecuentes de cada variable, con su porcentaje sobre el total de avisos.

In [327]:
categoricas = ["provincia", "area", "nivel_laboral", "modalidad", "tipo_trabajo", "contratacion", "educacion", "nivel_ingles"]

filas = []
for col in categoricas:
    for valor, n in limpio[col].value_counts().head(5).items():
        filas.append({"columna": col, "valor": valor, "avisos": n, "pct": round(n / len(limpio) * 100, 1)})
frecuencias = pd.DataFrame(filas)

with pd.option_context("display.max_rows", 60):
    display(frecuencias)

# Valor más frecuente de cada variable categórica y su peso
mas_frecuente = frecuencias.groupby("columna", sort=False).first().sort_values("pct")
mayoria = (mas_frecuente["pct"] > 50).sum()
mas_frecuente["etiqueta"] = mas_frecuente["valor"].astype(str) + " · " + mas_frecuente["pct"].astype(str) + "%"
fig = px.bar(mas_frecuente, x="pct", y=mas_frecuente.index, orientation="h", text="etiqueta",
             title=f"En {mayoria} de {len(mas_frecuente)} variables, un solo valor reúne más de la mitad de los avisos",
             labels={"pct": "Avisos con el valor más frecuente (%)", "columna": ""})
fig.update_xaxes(range=[0, 100])
fig.show()

def pct_valor(col, valor):
    """Porcentaje de avisos con ese valor en la columna."""
    return (limpio[col] == valor).mean() * 100

educacion_conocida = limpio.loc[limpio["educacion"] != "Sin dato", "educacion"].value_counts(normalize=True)
interpretar(f"El empleo ofrecido es casi todo formal y estable: {pct_valor('tipo_trabajo', 'Full-time'):.1f}% es a tiempo completo "
            f"y {pct_valor('contratacion', 'Indeterminado'):.1f}% por tiempo indeterminado. Entre los avisos que indican "
            f"estudios, el más pedido es «{educacion_conocida.index[0]}» ({educacion_conocida.iloc[0]:.0%}); "
            f"pero el {pct_valor('educacion', 'Sin dato'):.0f}% no indica ninguno.", "7.3", fig, tablas=[frecuencias])

,columna,valor,avisos,pct
0,provincia,Panamá,2312,78.1
1,provincia,Colón,148,5.0
2,provincia,Chiriquí,142,4.8
3,provincia,Panamá Oeste,121,4.1
4,provincia,Coclé,63,2.1
5,area,"Comercial, Ventas y Negocios",783,26.5
6,area,"Administración, Contabilidad y Finanzas",442,14.9
7,area,Abastecimiento y Logística,290,9.8
8,area,Oficios y Otros,223,7.5
9,area,"Tecnología, Sistemas y Telecomunicaciones",144,4.9


**Interpretación:** El empleo ofrecido es casi todo formal y estable: 99.1% es a tiempo completo y 75.0% por tiempo indeterminado. Entre los avisos que indican estudios, el más pedido es «Universitario» (61%); pero el 60% no indica ninguno.

### 7.4 Distribuciones

In [328]:
# Distribución del salario medio publicado
bajo_1500 = (con_salario["salario_medio"] < 1500).mean()
mediana_salario = con_salario["salario_medio"].median()

fig = px.histogram(con_salario, x="salario_medio", nbins=40,
                   title=f"El {bajo_1500:.0%} de los salarios publicados es menor a B/. 1,500 mensuales",
                   labels={"salario_medio": "Salario medio (B/. mensuales brutos)"})
fig.update_layout(yaxis_title="N.º de avisos", bargap=0.05)
fig.add_vline(x=mediana_salario, line_dash="dash", annotation_text=f"Mediana: B/. {mediana_salario:,.0f}")
fig.show()

entre = con_salario["salario_medio"].between(600, 1200).mean()
interpretar(f"La distribución tiene una cola larga hacia la derecha. El {entre:.0%} de los salarios está entre B/. 600 y "
            f"B/. 1,200, y solo el {(con_salario['salario_medio'] > 2000).mean():.0%} supera los B/. 2,000 (n = {len(con_salario)} avisos con salario).", "7.4", fig)

**Interpretación:** La distribución tiene una cola larga hacia la derecha. El 65% de los salarios está entre B/. 600 y B/. 1,200, y solo el 7% supera los B/. 2,000 (n = 442 avisos con salario).

In [329]:
# Distribución de la experiencia pedida
experiencia = limpio["experiencia_meses"].value_counts().sort_index()
moda_experiencia = experiencia.idxmax()

fig = px.bar(x=experiencia.index, y=experiencia.values,
             title=f"Lo más pedido es {moda_experiencia:.0f} meses de experiencia",
             labels={"x": "Experiencia pedida (meses)", "y": "N.º de avisos"})
fig.update_xaxes(tickvals=experiencia.index)
fig.show()

hasta_2_anios = (limpio["experiencia_meses"] <= 24).sum() / limpio["experiencia_meses"].count()
interpretar(f"Solo el {limpio['pide_experiencia'].mean():.0%} de los avisos pide experiencia, y de esos el {hasta_2_anios:.0%} "
            f"pide 2 años o menos (n = {limpio['experiencia_meses'].count()} avisos con experiencia).", "7.4", fig)

**Interpretación:** Solo el 28% de los avisos pide experiencia, y de esos el 67% pide 2 años o menos (n = 840 avisos con experiencia).

### 7.5 Comparaciones entre grupos
Salario por área, quién publica salario según el nivel y requisitos por nivel. Solo se comparan los salarios de las áreas con al menos `MIN_AVISOS_SALARIO` avisos con salario (`n_suficiente` en `t4`).

In [330]:
# Salario por área (solo áreas con suficientes salarios)
areas_comparables = t4_salario_area[t4_salario_area["n_suficiente"]].sort_values("salario_mediana", ascending=False)
datos_area = con_salario[con_salario["area"].isin(areas_comparables["area"])]
mas_alta, mas_baja = areas_comparables.iloc[0], areas_comparables.iloc[-1]

fig = px.box(datos_area, x="salario_medio", y="area", points=False,
             category_orders={"area": list(areas_comparables["area"])},
             title=f"{mas_alta['area']} tiene la mediana salarial más alta (B/. {mas_alta['salario_mediana']:,.0f})",
             labels={"salario_medio": "Salario medio (B/. mensuales brutos)", "area": ""})
fig.update_layout(height=550)
fig.show()

interpretar(f"Entre las {len(areas_comparables)} áreas con al menos {MIN_AVISOS_SALARIO} salarios, la mediana va de "
            f"B/. {mas_baja['salario_mediana']:,.0f} ({mas_baja['area']}) a B/. {mas_alta['salario_mediana']:,.0f} "
            f"({mas_alta['area']}): {mas_alta['salario_mediana'] / mas_baja['salario_mediana']:.1f} veces más "
            f"(n = {len(datos_area)} avisos con salario).", "7.5", fig)

**Interpretación:** Entre las 13 áreas con al menos 10 salarios, la mediana va de B/. 700 (Gastronomía y Turismo) a B/. 1,492 (Tecnología, Sistemas y Telecomunicaciones): 2.1 veces más (n = 397 avisos con salario).

In [331]:
# ¿En qué niveles están los avisos que publican salario?
fig = px.bar(t5_salario_nivel, x="nivel_laboral", y="con_salario", text="con_salario",
             title=f"El {pct_semi_o_sin_nivel:.0%} de los salarios publicados viene de avisos Semi Sr o sin nivel",
             labels={"nivel_laboral": "Nivel laboral", "con_salario": "N.º de avisos con salario"})
fig.show()

no_comparables = t5_salario_nivel.loc[~t5_salario_nivel["n_suficiente"], "nivel_laboral"].tolist()
interpretar(f"Los niveles {', '.join(no_comparables)} tienen menos de {MIN_AVISOS_SALARIO} salarios publicados cada uno, así que "
            "**no se puede comparar el salario entre niveles**: la mediana de un grupo tan pequeño no es representativa.", "7.5", fig)

**Interpretación:** Los niveles Sin Experiencia, Trainee / Pasante, Senior, Jefe / Supervisor / Responsable, Gerencia / Alta Gerencia / Dirección, Otro tienen menos de 10 salarios publicados cada uno, así que **no se puede comparar el salario entre niveles**: la mediana de un grupo tan pequeño no es representativa.

In [332]:
# Requisitos por nivel (solo niveles de la escala jerárquica)
requisitos = t6_requisitos_nivel[t6_requisitos_nivel["nivel_laboral"].isin(ESCALA_NIVEL)]
nombres_requisito = {"pct_pide_experiencia": "Pide experiencia", "pct_pide_ingles": "Pide inglés",
                     "pct_fija_edad": "Fija edad", "pct_universitario_o_mas": "Universitario o más"}
requisitos_largo = requisitos.melt(id_vars="nivel_laboral", value_vars=list(nombres_requisito),
                                   var_name="requisito", value_name="pct")
requisitos_largo["requisito"] = requisitos_largo["requisito"].map(nombres_requisito)

req = t6_requisitos_nivel.set_index("nivel_laboral")
# Extremos de la escala: primer y último nivel con suficientes avisos
NIVELES_CORTOS = {"Jefe / Supervisor / Responsable": "Jefe", "Gerencia / Alta Gerencia / Dirección": "Gerencia"}
niveles_validos = requisitos.loc[requisitos["avisos"] >= MIN_AVISOS_AREA, "nivel_laboral"].astype(str).tolist()
req_bajo, req_alto = req.loc[niveles_validos[0]], req.loc[niveles_validos[-1]]
nombre_bajo, nombre_alto = [NIVELES_CORTOS.get(n, n) for n in (niveles_validos[0], niveles_validos[-1])]

fig = px.bar(requisitos_largo, x="nivel_laboral", y="pct", color="requisito", barmode="group",
             title=f"Los niveles altos piden más estudios: universitario de {req_bajo['pct_universitario_o_mas']:.0f}% ({nombre_bajo}) "
                   f"a {req_alto['pct_universitario_o_mas']:.0f}% ({nombre_alto})",
             labels={"nivel_laboral": "Nivel laboral", "pct": "% de avisos del nivel", "requisito": "Requisito"})
fig.show()

interpretar(f"De {nombre_bajo} a {nombre_alto}, pedir inglés pasa de {req_bajo['pct_pide_ingles']:.0f}% a {req_alto['pct_pide_ingles']:.0f}% "
            f"y pedir experiencia de {req_bajo['pct_pide_experiencia']:.0f}% a {req_alto['pct_pide_experiencia']:.0f}%. "
            f"Aun así, incluso en {nombre_alto} el {100 - req_alto['pct_pide_ingles']:.0f}% de los avisos no pide inglés de forma explícita "
            f"(n = {requisitos['avisos'].sum():,} avisos con nivel).", "7.5", fig)

**Interpretación:** De Junior a Gerencia, pedir inglés pasa de 9% a 27% y pedir experiencia de 28% a 38%. Aun así, incluso en Gerencia el 73% de los avisos no pide inglés de forma explícita (n = 2,604 avisos con nivel).

### 7.6 Relaciones entre variables
Se usa la correlación de **Spearman**, que mide si dos variables suben o bajan juntas sin suponer una relación lineal y es menos sensible a valores extremos. Cada par se calcula con los avisos que tienen ambos datos, así que el número de avisos cambia de un par a otro.

In [333]:
# Matriz de correlación entre variables numéricas
correlaciones = limpio[numericas].corr(method="spearman").round(2)
MIN_AVISOS_PAR = 30   # mínimo de avisos con ambos datos para tomar en serio una correlación

# Avisos con ambos datos en cada par de variables
presentes = limpio[numericas].notna().astype(int)
avisos_par = presentes.T @ presentes

pares = correlaciones.mask(np.eye(len(correlaciones), dtype=bool)).stack()
pares = pares[[a < b for a, b in pares.index]]                                  # cada par una sola vez
pares_confiables = pares[[avisos_par.loc[a, b] >= MIN_AVISOS_PAR for a, b in pares.index]]
par_fuerte = pares_confiables.abs().idxmax()
n_par = avisos_par.loc[par_fuerte]

# Cada par con su ρ y sus avisos, del ρ más fuerte al más débil
tabla_pares = pd.DataFrame([{"variable 1": a, "variable 2": b, "ρ de Spearman": rho, "avisos con ambos datos": avisos_par.loc[a, b],
                             "¿confiable?": "Sí" if avisos_par.loc[a, b] >= MIN_AVISOS_PAR else "No"}
                            for (a, b), rho in pares.items()])
tabla_pares = tabla_pares.sort_values("ρ de Spearman", key=abs, ascending=False).reset_index(drop=True)

fig = px.imshow(correlaciones, text_auto=True, color_continuous_scale="RdBu", zmin=-1, zmax=1,
                title=f"Entre los pares con datos suficientes, la relación más fuerte es {par_fuerte[0]} y {par_fuerte[1]} (ρ = {pares[par_fuerte]:.2f})",
                labels={"color": "ρ de Spearman"})
fig.show()

n_salario_experiencia = limpio[["salario_medio", "experiencia_meses"]].dropna().shape[0]
display(tabla_pares)
display(avisos_par.rename_axis("avisos con ambos datos"))
par_mayor = tabla_pares.iloc[0]
aviso_mayor = (f"El ρ más alto de la matriz ({par_mayor['ρ de Spearman']:.2f}, {par_mayor['variable 1']} y {par_mayor['variable 2']}) no se toma en cuenta "
               f"porque solo tiene {par_mayor['avisos con ambos datos']} avisos con ambos datos. " if par_mayor["¿confiable?"] == "No" else "")
sentido = "más alto es uno, más alto tiende a ser el otro" if pares[par_fuerte] > 0 else "más alto es uno, más bajo tiende a ser el otro"
interpretar(f"La correlación entre {par_fuerte[0]} y {par_fuerte[1]} se calcula con {n_par} avisos: {sentido}. {aviso_mayor}Las correlaciones del salario con los requisitos no son "
            f"confiables: salario y experiencia aparecen juntos en solo {n_salario_experiencia} avisos, porque los avisos que "
            f"publican salario casi no piden requisitos (se exigen al menos {MIN_AVISOS_PAR} avisos por par).", "7.6", fig, [tabla_pares, avisos_par.rename_axis("avisos con ambos datos")])

,variable 1,variable 2,ρ de Spearman,avisos con ambos datos,¿confiable?
0,edad_min,salario_medio,0.66,9,No
1,edad_min,experiencia_meses,0.58,549,Sí
2,edad_min,vacantes,-0.39,651,Sí
3,edad_max,edad_min,0.39,648,Sí
4,edad_max,experiencia_meses,0.28,546,Sí
5,experiencia_meses,vacantes,-0.24,840,Sí
6,edad_max,salario_medio,-0.20,9,No
7,experiencia_meses,salario_medio,-0.12,12,No
8,edad_max,vacantes,-0.08,648,Sí
9,salario_medio,vacantes,0.00,442,Sí


,vacantes,salario_medio,experiencia_meses,edad_min,edad_max
avisos con ambos datos,,,,,
vacantes,2960,442,840,651,648
salario_medio,442,442,12,9,9
experiencia_meses,840,12,840,549,546
edad_min,651,9,549,651,648
edad_max,648,9,546,648,648


**Interpretación:** La correlación entre edad_min y experiencia_meses se calcula con 549 avisos: más alto es uno, más alto tiende a ser el otro. El ρ más alto de la matriz (0.66, edad_min y salario_medio) no se toma en cuenta porque solo tiene 9 avisos con ambos datos. Las correlaciones del salario con los requisitos no son confiables: salario y experiencia aparecen juntos en solo 12 avisos, porque los avisos que publican salario casi no piden requisitos (se exigen al menos 30 avisos por par).

In [334]:
# Experiencia pedida según el nivel
con_experiencia = limpio[limpio["pide_experiencia"] & limpio["nivel_laboral"].isin(ESCALA_NIVEL)]

fig = px.box(con_experiencia, x="nivel_laboral", y="experiencia_meses",
             category_orders={"nivel_laboral": ESCALA_NIVEL},
             title=f"La experiencia pedida crece con el nivel: de {req_bajo['experiencia_mediana_meses']:.0f} meses ({nombre_bajo}) "
                   f"a {req_alto['experiencia_mediana_meses']:.0f} ({nombre_alto})",
             labels={"nivel_laboral": "Nivel laboral", "experiencia_meses": "Experiencia pedida (meses)"})
fig.show()

interpretar(f"La mediana de experiencia pedida se multiplica por "
            f"{req_alto['experiencia_mediana_meses'] / req_bajo['experiencia_mediana_meses']:.0f} de {nombre_bajo} a {nombre_alto} "
            f"(n = {len(con_experiencia)} avisos que piden experiencia).", "7.6", fig)

**Interpretación:** La mediana de experiencia pedida se multiplica por 5 de Junior a Gerencia (n = 787 avisos que piden experiencia).

In [335]:
# Publicación de salario según si la empresa es confidencial
publica_confidencial = (limpio.groupby("empresa_confidencial")["publica_salario"].mean() * 100).round(1)
publica_confidencial = publica_confidencial.reindex([False, True], fill_value=0)
publica_confidencial.index = ["Empresa identificada", "Confidencial"]

fig = px.bar(x=publica_confidencial.index, y=publica_confidencial.values, text=publica_confidencial.values,
             title=f"Las empresas confidenciales casi nunca publican salario ({publica_confidencial['Confidencial']}%)",
             labels={"x": "", "y": "% de avisos que publica salario"})
fig.show()

veces = publica_confidencial["Empresa identificada"] / publica_confidencial["Confidencial"]
interpretar(f"Una empresa identificada publica el salario {veces:.0f} veces más que una confidencial "
            f"({publica_confidencial['Empresa identificada']}% frente a {publica_confidencial['Confidencial']}%): "
            "la falta de transparencia se acumula en los mismos avisos.", "7.6", fig)

**Interpretación:** Una empresa identificada publica el salario 30 veces más que una confidencial (23.8% frente a 0.8%): la falta de transparencia se acumula en los mismos avisos.

In [336]:
# Vacantes por aviso según el área (contrataciones masivas)
vacantes_area = t1_demanda_area.assign(vacantes_por_aviso=(t1_demanda_area["vacantes"] / t1_demanda_area["avisos"]).round(1))
vacantes_area = vacantes_area[vacantes_area["avisos"] >= 30].sort_values("vacantes_por_aviso", ascending=False)
display(vacantes_area[["area", "avisos", "vacantes", "vacantes_por_aviso"]].head(5))

masiva = vacantes_area.iloc[0]
interpretar(f"{masiva['area']} ofrece {masiva['vacantes_por_aviso']} vacantes por aviso, frente a "
            f"{limpio['vacantes'].sum() / len(limpio):.1f} en promedio: es el área con contrataciones más masivas "
            "(se consideran las áreas con al menos 30 avisos).", "7.6", tablas=[vacantes_area[["area", "avisos", "vacantes", "vacantes_por_aviso"]].head(5)])

,area,avisos,vacantes,vacantes_por_aviso
7,"Atención al Cliente, Call Center y Telemarketing",137,1158,8.5
0,"Comercial, Ventas y Negocios",783,2713,3.5
3,Oficios y Otros,223,605,2.7
5,Gastronomía y Turismo,142,313,2.2
2,Abastecimiento y Logística,290,615,2.1


**Interpretación:** Atención al Cliente, Call Center y Telemarketing ofrece 8.5 vacantes por aviso, frente a 2.5 en promedio: es el área con contrataciones más masivas (se consideran las áreas con al menos 30 avisos).

### 7.7 Tendencia en el tiempo
Las zonas grises marcan las semanas **no comparables** (ver 6.5): las antiguas, en las que muchos avisos ya habían vencido al momento de la extracción, y la última, que está incompleta.

In [337]:
# Avisos por semana, marcando las semanas no comparables
comparables = t7_semanal[t7_semanal["semana_comparable"]]
por_mes = comparables.groupby(comparables["semana"].dt.month)["avisos"].mean()
mes_inicial, mes_final = por_mes.index[0], por_mes.index[-1]

fig = px.line(t7_semanal, x="semana", y="avisos", markers=True,
              title=f"Entre semanas comparables, {MESES[mes_final]} promedia {por_mes[mes_final]:.0f} avisos por semana "
                    f"frente a {por_mes[mes_inicial]:.0f} en {MESES[mes_inicial]}",
              labels={"semana": "Semana de publicación", "avisos": "N.º de avisos publicados"})
for _, fila in t7_semanal[~t7_semanal["semana_comparable"]].iterrows():
    fig.add_vrect(x0=fila["semana"], x1=fila["semana"] + pd.Timedelta(days=7), fillcolor="gray", opacity=0.15, line_width=0)
fig.show()

no_comparables_antes = t7_semanal[~t7_semanal["semana_comparable"] & (t7_semanal["semana"] < inicio_comparable)]
interpretar(f"Las semanas anteriores al {inicio_comparable:%d/%m} promedian {no_comparables_antes['avisos'].mean():.0f} avisos, "
            f"pero no reflejan menos demanda: los avisos duran unos {duracion_tipica:.0f} días y la mayoría ya había vencido. "
            f"Entre las semanas comparables sí hay un aumento de {MESES[mes_inicial]} a {MESES[mes_final]} "
            f"({(por_mes[mes_final] / por_mes[mes_inicial] - 1):+.0%}).", "7.7", fig)

**Interpretación:** Las semanas anteriores al 09/08 promedian 28 avisos, pero no reflejan menos demanda: los avisos duran unos 60 días y la mayoría ya había vencido. Entre las semanas comparables sí hay un aumento de agosto a septiembre (+19%).

In [338]:
# Avisos por día de la semana
dias = limpio["fecha_publicacion"].dt.day_of_week.map(dict(enumerate(DIAS)))
por_dia = dias.value_counts().reindex(DIAS, fill_value=0)

fig = px.bar(x=por_dia.index, y=por_dia.values,
             title=f"El {por_dia[DIAS[:5]].sum() / por_dia.sum():.0%} de los avisos se publica de lunes a viernes",
             labels={"x": "Día de publicación", "y": "N.º de avisos"})
fig.show()

interpretar(f"Los días con más publicaciones son {por_dia.idxmax()} ({por_dia.max()} avisos) y "
            f"{por_dia.drop(por_dia.idxmax()).idxmax()}: las empresas publican al inicio de la semana laboral.", "7.7", fig)

**Interpretación:** Los días con más publicaciones son martes (647 avisos) y lunes: las empresas publican al inicio de la semana laboral.

### 7.8 Hallazgos
Resumen de lo encontrado, con la cifra que lo sustenta. El texto se genera desde los datos.

In [339]:
area_top = t1_demanda_area.iloc[0]
top3_areas = t1_demanda_area.iloc[2]["pct_acumulado"]
provincia_top, provincia_2 = t2_demanda_provincia.iloc[0], t2_demanda_provincia.iloc[1]
industria_top = t3_demanda_industria.iloc[0]
presencial = t8_modalidad.set_index(t8_modalidad["modalidad"].astype(str))["pct_avisos"].reindex(["Presencial", "Híbrido", "Remoto"], fill_value=0)

hallazgos = [
    f"**Demanda concentrada en pocas áreas.** {area_top['area']} reúne el {area_top['pct_avisos']}% de los avisos y el "
    f"{area_top['pct_vacantes']}% de las vacantes; las 3 primeras áreas suman el {top3_areas}%.",
    f"**Demanda concentrada en la provincia de Panamá:** {provincia_top['pct_avisos']}% de los avisos. La sigue "
    f"{provincia_2['provincia']} con apenas {provincia_2['pct_avisos']}%.",
    f"**{industria_top['industria']}** es la industria con más avisos ({industria_top['pct_avisos']}% de los que indican "
    f"industria), aunque solo el {len(con_industria) / len(limpio):.0%} de los avisos la indica.",
    f"**Poca transparencia salarial:** solo el {limpio['publica_salario'].mean():.1%} publica un salario válido; en las "
    f"empresas confidenciales, el {publica_confidencial['Confidencial']}%.",
    f"**Salario típico de B/. {mediana_salario:,.0f} mensuales brutos** (mediana), con diferencias de "
    f"{mas_alta['salario_mediana'] / mas_baja['salario_mediana']:.1f} veces entre áreas: de B/. {mas_baja['salario_mediana']:,.0f} "
    f"({mas_baja['area']}) a B/. {mas_alta['salario_mediana']:,.0f} ({mas_alta['area']}).",
    f"**El salario por nivel no se puede comparar:** el {pct_semi_o_sin_nivel:.0%} de los salarios publicados "
    "viene de avisos Semi Sr o sin nivel.",
    f"**Los requisitos crecen con el nivel:** de {nombre_bajo} a {nombre_alto}, la experiencia mediana pasa de "
    f"{req_bajo['experiencia_mediana_meses']:.0f} a {req_alto['experiencia_mediana_meses']:.0f} meses, el inglés de "
    f"{req_bajo['pct_pide_ingles']:.0f}% a {req_alto['pct_pide_ingles']:.0f}% y los estudios universitarios de "
    f"{req_bajo['pct_universitario_o_mas']:.0f}% a {req_alto['pct_universitario_o_mas']:.0f}%.",
    f"**El {limpio['fija_edad'].mean():.0%} de los avisos fija un rango de edad**, con edad máxima mediana de "
    f"{limpio['edad_max'].median():.0f} años.",
    f"**La publicación crece hacia {MESES[mes_final]}** entre semanas comparables ({(por_mes[mes_final] / por_mes[mes_inicial] - 1):+.0%} "
    f"frente a {MESES[mes_inicial]}); las semanas anteriores al {inicio_comparable:%d/%m} están subrepresentadas por la forma de extracción.",
    f"**Trabajo casi totalmente presencial:** {presencial['Presencial']}%; híbrido y remoto suman "
    f"{presencial.drop('Presencial').sum():.1f}%.",
]

display(Markdown("\n".join(f"{i}. {texto}" for i, texto in enumerate(hallazgos, start=1))))

1. **Demanda concentrada en pocas áreas.** Comercial, Ventas y Negocios reúne el 26.5% de los avisos y el 36.3% de las vacantes; las 3 primeras áreas suman el 51.2%.
2. **Demanda concentrada en la provincia de Panamá:** 78.1% de los avisos. La sigue Colón con apenas 5.0%.
3. **Banca / Financiera** es la industria con más avisos (13.1% de los que indican industria), aunque solo el 60% de los avisos la indica.
4. **Poca transparencia salarial:** solo el 14.9% publica un salario válido; en las empresas confidenciales, el 0.8%.
5. **Salario típico de B/. 875 mensuales brutos** (mediana), con diferencias de 2.1 veces entre áreas: de B/. 700 (Gastronomía y Turismo) a B/. 1,492 (Tecnología, Sistemas y Telecomunicaciones).
6. **El salario por nivel no se puede comparar:** el 94% de los salarios publicados viene de avisos Semi Sr o sin nivel.
7. **Los requisitos crecen con el nivel:** de Junior a Gerencia, la experiencia mediana pasa de 12 a 60 meses, el inglés de 9% a 27% y los estudios universitarios de 21% a 60%.
8. **El 22% de los avisos fija un rango de edad**, con edad máxima mediana de 43 años.
9. **La publicación crece hacia septiembre** entre semanas comparables (+19% frente a agosto); las semanas anteriores al 09/08 están subrepresentadas por la forma de extracción.
10. **Trabajo casi totalmente presencial:** 97.7%; híbrido y remoto suman 2.2%.

### 7.9 Contraste con lo esperado en la propuesta
Las hipótesis vienen de la propuesta; el resultado y la conclusión se calculan con los datos limpios. Una cifra se considera **cumplida** si queda a menos de 3 puntos porcentuales (o 10 % en el salario) de lo esperado.

In [340]:
def comparar(valor, esperado, tolerancia):
    """«Sí» si el valor está dentro de la tolerancia; si no, indica si salió mayor o menor."""
    if abs(valor - esperado) <= tolerancia:
        return "Sí"
    return "No: mayor" if valor > esperado else "No: menor"

def pct_de(tabla, columna, valor):
    """% de avisos de un valor en una tabla de demanda; 0 si el valor no aparece."""
    return float(tabla.loc[tabla[columna] == valor, "pct_avisos"].sum())

senior_n = int(t5_salario_nivel.loc[t5_salario_nivel["nivel_laboral"] == "Senior", "con_salario"].sum())
crece_nivel = (req_alto["experiencia_mediana_meses"] > req_bajo["experiencia_mediana_meses"]) and \
              (req_alto["pct_pide_ingles"] > req_bajo["pct_pide_ingles"])

contraste = pd.DataFrame([
    ("P1", "«Comercial, Ventas y Negocios» ≈ 27 % de los avisos", f"{pct_de(t1_demanda_area, 'area', 'Comercial, Ventas y Negocios')}%",
     comparar(pct_de(t1_demanda_area, "area", "Comercial, Ventas y Negocios"), 27, 3)),
    ("P1", "Provincia de Panamá ≈ 78 % de los avisos", f"{pct_de(t2_demanda_provincia, 'provincia', 'Panamá')}%",
     comparar(pct_de(t2_demanda_provincia, "provincia", "Panamá"), 78, 3)),
    ("P2", "Solo ≈ 16 % publica salario", f"{limpio['publica_salario'].mean() * 100:.1f}%",
     comparar(limpio["publica_salario"].mean() * 100, 16, 3)),
    ("P2", "Mediana salarial ≈ B/. 1,000", f"B/. {mediana_salario:,.0f}", comparar(mediana_salario, 1000, 100)),
    ("P2", "Salto claro de salario entre Junior, Semi Sr y Senior", f"Senior: solo {senior_n} salarios publicados",
     "No se puede comprobar" if senior_n < MIN_AVISOS_SALARIO else "Revisar"),
    ("P3", "Más experiencia e inglés en niveles altos",
     f"Experiencia {req_bajo['experiencia_mediana_meses']:.0f} → {req_alto['experiencia_mediana_meses']:.0f} meses; "
     f"inglés {req_bajo['pct_pide_ingles']:.0f}% → {req_alto['pct_pide_ingles']:.0f}%", "Sí" if crece_nivel else "No"),
    ("P3", "≈ 22 % de los avisos fija un rango de edad", f"{limpio['fija_edad'].mean() * 100:.1f}%",
     comparar(limpio["fija_edad"].mean() * 100, 22, 3)),
    ("P4", "Crecimiento de la publicación hacia septiembre",
     f"{MESES[mes_final]}: {por_mes[mes_final]:.0f} vs. {MESES[mes_inicial]}: {por_mes[mes_inicial]:.0f} avisos/semana (comparables)",
     "Sí, pero menor de lo que parece" if por_mes[mes_final] > por_mes[mes_inicial] else "No"),
    ("P4", "Modalidad casi totalmente presencial (≈ 98 %)", f"{presencial['Presencial']}%", comparar(presencial["Presencial"], 98, 3)),
], columns=["pregunta", "esperado (propuesta)", "resultado", "¿se cumple?"])

with pd.option_context("display.max_colwidth", None):
    display(contraste)

,pregunta,esperado (propuesta),resultado,¿se cumple?
0,P1,"«Comercial, Ventas y Negocios» ≈ 27 % de los avisos",26.5%,Sí
1,P1,Provincia de Panamá ≈ 78 % de los avisos,78.1%,Sí
2,P2,Solo ≈ 16 % publica salario,14.9%,Sí
3,P2,"Mediana salarial ≈ B/. 1,000",B/. 875,No: menor
4,P2,"Salto claro de salario entre Junior, Semi Sr y Senior",Senior: solo 3 salarios publicados,No se puede comprobar
5,P3,Más experiencia e inglés en niveles altos,Experiencia 12 → 60 meses; inglés 9% → 27%,Sí
6,P3,≈ 22 % de los avisos fija un rango de edad,22.0%,Sí
7,P4,Crecimiento de la publicación hacia septiembre,septiembre: 353 vs. agosto: 296 avisos/semana (comparables),"Sí, pero menor de lo que parece"
8,P4,Modalidad casi totalmente presencial (≈ 98 %),97.7%,Sí


## 8. Dashboard
Esta sección genera `reporte/dashboard/index.html` en cada ejecución. La página es un reportaje de datos interactivo construido desde `limpio` y las tablas de la sección 6: todas las cifras, títulos e interpretaciones cambian solos si cambian los datos originales.

Es un solo archivo HTML con Plotly incluido, así que se abre en cualquier navegador y sin internet.

**Historia del dashboard:**

| Parte | Pregunta | Contenido |
|---|---|---|
| Portada | ¿Cómo es el mercado en una mirada? | «Panamá en 100 avisos»: cuadrícula de 100 cuadros |
| Capítulo 1 · Contexto | ¿Dónde está la demanda? | Avisos por área · Avisos por industria · Mapa esquemático de provincias |
| Capítulo 2 · Hallazgos | ¿Qué ofrecen las empresas? | Indicadores de transparencia · Salario por área |
| Capítulo 3 · Hallazgos | ¿Qué piden las empresas? | Requisitos por nivel |
| Capítulo 4 · Hallazgos | ¿Cuándo y cómo se contrata? | Avisos por semana · Híbrido o remoto por área |
| Capítulo 5 | ¿Cómo es mi área? | Explorador interactivo por área |

La pestaña **Análisis exploratorio** muestra aparte el EDA completo de la sección 7, con sus tablas, gráficos, hallazgos y contraste. La pestaña **Historia** conserva solo lo necesario para contar el mercado.

Las conclusiones, las recomendaciones y las limitaciones van en la presentación.

### 8.1 Estilo de los gráficos
Una sola serie usa un solo color; el gris deja en segundo plano lo que no es el foco. La paleta se validó para personas con daltonismo. Debajo de cada gráfico, la página incluye una tabla con los datos.

In [341]:
from datetime import datetime
from html import escape
import json
import re
import textwrap
from plotly.offline import get_plotlyjs
import plotly.graph_objects as go

AZUL, NARANJA, AQUA, AMARILLO = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"   # paleta categórica validada
GRIS = "#b9b8b1"                                                              # marcas en segundo plano

def envolver(texto, ancho=28):
    """Parte un nombre largo en varias líneas para que el eje no le quite espacio al gráfico."""
    return "<br>".join(textwrap.wrap(str(texto), ancho))

def estilo_grafico(fig, alto=420, leyenda=False):
    """Aplica el estilo común: fondo transparente, rejilla suave y márgenes automáticos."""
    fig.update_layout(
        height=alto, showlegend=leyenda, paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
        margin=dict(l=10, r=30, t=30, b=50), font=dict(size=13, color="#52514e", family='system-ui, -apple-system, "Segoe UI", Roboto, sans-serif'), barcornerradius=4,
        hoverlabel=dict(align="left"), legend=dict(orientation="h", y=1.02, yanchor="bottom", x=0),
    )
    fig.update_xaxes(gridcolor="#e6e5e0", zeroline=False, automargin=True)
    fig.update_yaxes(gridcolor="#e6e5e0", zeroline=False, automargin=True)
    return fig

def barras_horizontales(tabla, categoria, valor, resaltar, eje_x, formato, plantilla, alto=420):
    """Barras horizontales de una serie: resalta las primeras filas y etiqueta solo esas."""
    orden = tabla.iloc[::-1]                     # Plotly dibuja la primera fila abajo
    resaltada = orden.index.isin(tabla.index[:resaltar])
    fig = go.Figure(go.Bar(
        x=orden[valor], y=orden[categoria].map(envolver).astype(str), orientation="h",
        marker_color=[AZUL if r else GRIS for r in resaltada],
        text=[formato(v) if r else "" for v, r in zip(orden[valor], resaltada)],
        textposition="outside", cliponaxis=False, hovertemplate=plantilla + "<extra></extra>",
    ))
    fig.update_xaxes(title=eje_x)
    return estilo_grafico(fig, alto)

### 8.2 Gráficos
Cada gráfico se guarda en una variable para insertarlo en la página en 8.4. Los títulos y las interpretaciones se calculan en esa misma sección.

In [342]:
# 1. Avisos por área (las 10 con más avisos)
fig_area = barras_horizontales(t1_demanda_area.head(10), "area", "avisos", 3, "N.º de avisos publicados",
                               lambda v: f"{v:,}", "%{y}<br>%{x:,} avisos")
fig_area.show()

In [343]:
# 2. Avisos por industria (las 10 con más avisos, sin contar la categoría genérica «Otra»)
industrias_grafico = t3_demanda_industria[t3_demanda_industria["industria"] != "Otra"].head(10).reset_index(drop=True)
fig_industria = barras_horizontales(industrias_grafico, "industria", "avisos", 1, "N.º de avisos publicados",
                                    lambda v: f"{v:,}", "%{y}<br>%{x:,} avisos")
fig_industria.show()

In [344]:
# 4. Salario por área: el punto es la mediana y la línea va de Q1 a Q3
salario_grafico = areas_comparables.iloc[::-1]
extremos = [mas_alta["area"], mas_baja["area"]]

fig_salario = go.Figure(go.Scatter(
    x=salario_grafico["salario_mediana"], y=salario_grafico["area"].map(envolver).astype(str),
    mode="markers+text", marker=dict(size=10, color=AZUL),
    error_x=dict(type="data", symmetric=False, color=AZUL, thickness=2, width=0,
                 array=salario_grafico["salario_q3"] - salario_grafico["salario_mediana"],
                 arrayminus=salario_grafico["salario_mediana"] - salario_grafico["salario_q1"]),
    text=[f"B/. {m:,.0f}" if a in extremos else "" for a, m in zip(salario_grafico["area"], salario_grafico["salario_mediana"])],
    textposition="top center",
    customdata=salario_grafico[["salario_q1", "salario_q3", "con_salario", "pct_publica_salario"]],
    hovertemplate=("%{y}<br>Mediana: B/. %{x:,.0f}<br>Q1–Q3: B/. %{customdata[0]:,.0f} – %{customdata[1]:,.0f}"
                   "<br>%{customdata[2]} avisos con salario (%{customdata[3]}% del área)<extra></extra>"),
))
fig_salario.update_xaxes(title="Salario medio (B/. mensuales brutos)")
estilo_grafico(fig_salario, 600)
fig_salario.show()

In [345]:
# 5. Requisitos por nivel: solo los niveles jerárquicos con suficientes avisos
requisitos_grafico = requisitos[requisitos["avisos"] >= MIN_AVISOS_AREA]
eje_niveles = [NIVELES_CORTOS.get(n, n) for n in requisitos_grafico["nivel_laboral"].astype(str)]

series_requisito = [("pct_universitario_o_mas", "Universitario o más", AZUL),
                    ("pct_pide_experiencia", "Pide experiencia", NARANJA),
                    ("pct_fija_edad", "Fija edad", AQUA),
                    ("pct_pide_ingles", "Pide inglés", AMARILLO)]

fig_requisitos = go.Figure()
for columna, nombre, color in series_requisito:
    fig_requisitos.add_trace(go.Scatter(
        x=eje_niveles, y=requisitos_grafico[columna], name=nombre, mode="lines+markers",
        line=dict(color=color, width=2), marker=dict(size=9, color=color),
        hovertemplate=f"{nombre}<br>%{{x}}: %{{y:.1f}}% de los avisos<extra></extra>",
    ))
    # Etiqueta directa al final de cada línea
    fig_requisitos.add_annotation(x=eje_niveles[-1], y=requisitos_grafico[columna].iloc[-1], xshift=10,
                                  text=f"{nombre} {requisitos_grafico[columna].iloc[-1]:.0f}%",
                                  showarrow=False, xanchor="left")

fig_requisitos.update_xaxes(title="Nivel laboral")
fig_requisitos.update_yaxes(title="% de los avisos del nivel", rangemode="tozero")
estilo_grafico(fig_requisitos, 440, leyenda=True)
fig_requisitos.update_layout(margin=dict(r=170))
fig_requisitos.show()

In [346]:
# 6. Avisos por semana: las zonas grises son semanas no comparables (ver 6.5)
fig_semanal = go.Figure(go.Scatter(
    x=t7_semanal["semana"], y=t7_semanal["avisos"], mode="lines+markers",
    line=dict(color=AZUL, width=2), marker=dict(size=9, color=AZUL),
    hovertemplate="Semana del %{x|%d/%m/%Y}<br>%{y} avisos<extra></extra>",
))
for _, fila in t7_semanal[~t7_semanal["semana_comparable"]].iterrows():
    fig_semanal.add_vrect(x0=fila["semana"], x1=fila["semana"] + pd.Timedelta(days=7),
                          fillcolor="#8a8981", opacity=0.15, line_width=0)
fig_semanal.add_annotation(x=t7_semanal["semana"].iloc[0], y=1, yref="paper", xanchor="left", showarrow=False,
                           text="Gris: semanas no comparables")
fig_semanal.update_xaxes(title="Semana de publicación", tickformat="%d/%m")
fig_semanal.update_yaxes(title="N.º de avisos publicados", rangemode="tozero")
estilo_grafico(fig_semanal)
fig_semanal.show()

In [347]:
# 7. Avisos híbridos o remotos por área (solo áreas con alguno)
modalidad_grafico = t9_modalidad_area[t9_modalidad_area["hibrido_o_remoto"] > 0]
fig_modalidad = barras_horizontales(modalidad_grafico, "area", "pct_hibrido_o_remoto", 3, "% híbrido o remoto del área",
                                    lambda v: f"{v:.1f}%", "%{y}<br>%{x:.1f}% híbrido o remoto", alto=380)
fig_modalidad.show()

### 8.3 Piezas de la página
Funciones que arman el HTML. Todo texto calculado pasa por `escape()` para que un valor con símbolos especiales no rompa la página.

Los indicadores llevan su valor final en el HTML; la animación solo lo hace subir desde cero, así que la cifra se ve aunque el navegador no ejecute JavaScript.

In [348]:
def kpi(valor, etiqueta, prefijo="", sufijo="", decimales=0):
    """Indicador con cifra animada: el valor final ya viene escrito en el HTML."""
    texto = f"{prefijo}{valor:,.{decimales}f}{sufijo}"
    return (f'<div class="kpi"><strong data-valor="{valor}" data-decimales="{decimales}" data-prefijo="{escape(prefijo)}" '
            f'data-sufijo="{escape(sufijo)}">{escape(texto)}</strong><span>{escape(etiqueta)}</span></div>')

def tarjeta(titulo, fig, interpretacion, tabla, nota="", div_id=None):
    """Tarjeta de gráfico: título con el hallazgo, gráfico, interpretación, nota y tabla de datos."""
    grafico = fig.to_html(full_html=False, include_plotlyjs=False, div_id=div_id, config={"displayModeBar": False, "responsive": True})
    return (f'<section class="tarjeta aparece"><h3>{escape(titulo)}</h3>{grafico}'
            f'<p class="interpretacion">{escape(interpretacion)}</p>{nota_y_datos(nota, tabla)}</section>')

def nota_y_datos(nota, tabla):
    """Nota al pie y tabla desplegable con los datos."""
    nota_html = f'<p class="nota">{escape(nota)}</p>' if nota else ""
    tabla_html = tabla.to_html(index=False, border=0, na_rep="—")
    return f'{nota_html}<details><summary>Ver los datos</summary><div class="tabla">{tabla_html}</div></details>'

def capitulo(numero, ancla, titulo, pregunta, contenido):
    """Capítulo de la historia: número, título, pregunta y contenido."""
    return (f'<section class="capitulo" id="{ancla}"><div class="capitulo-cabeza aparece"><span class="numero">{numero}</span>'
            f'<h2>{escape(titulo)}</h2><p class="pregunta">{escape(pregunta)}</p></div>{"".join(contenido)}</section>')

In [349]:
# Mapa esquemático: cada provincia es un mosaico colocado según su ubicación (columna, fila)
POSICION_PROVINCIA = {"Bocas del Toro": (1, 1), "Chiriquí": (1, 2), "Veraguas": (2, 2), "Herrera": (2, 3), "Los Santos": (3, 3),
                      "Coclé": (3, 2), "Panamá Oeste": (4, 2), "Colón": (5, 1), "Panamá": (5, 2), "Darién": (6, 2)}
RAMPA_AZUL = ["#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec", "#5598e7", "#3987e5",
              "#2a78d6", "#256abf", "#1c5cab", "#184f95", "#104281", "#0d366b"]

def mapa_provincias(tabla):
    """Mosaico de provincias; el color sigue una escala logarítmica porque Panamá domina por mucho."""
    maximo = tabla["avisos"].max()
    mosaicos = []
    for _, fila in tabla[tabla["provincia"].isin(POSICION_PROVINCIA)].iterrows():
        columna, renglon = POSICION_PROVINCIA[fila["provincia"]]
        paso = round(np.log(fila["avisos"]) / np.log(maximo) * (len(RAMPA_AZUL) - 1))
        tinta = "#ffffff" if paso >= 7 else "#0b0b0b"
        mosaicos.append(f'<div class="provincia" style="--c:{columna};--r:{renglon};background:{RAMPA_AZUL[paso]};color:{tinta}">'
                        f'<b>{escape(str(fila["provincia"]))}</b><span>{fila["avisos"]:,}</span><small>{fila["pct_avisos"]}%</small></div>')
    return f'<div class="mapa" role="img" aria-label="Mapa esquemático de avisos por provincia">{"".join(mosaicos)}</div>'

fuera_del_mapa = t2_demanda_provincia[~t2_demanda_provincia["provincia"].isin(POSICION_PROVINCIA)]
leyenda_mapa = (f'<div class="escala"><span>Menos avisos</span><i style="background:linear-gradient(90deg,{",".join(RAMPA_AZUL)})"></i>'
                '<span>Más avisos (escala logarítmica)</span></div>')

### 8.4 Textos e indicadores
Indicadores y hechos de la portada. Todas las cifras vienen de los datos; lo único fijo es la redacción.

In [350]:
inicio_periodo, fin_periodo = limpio["fecha_publicacion"].min(), limpio["fecha_publicacion"].max()
modalidad_top = t9_modalidad_area.iloc[0]
pct_panama = provincia_top["pct_avisos"]

indicadores = "".join([
    kpi(len(limpio), "avisos analizados"),
    kpi(int(limpio["vacantes"].sum()), "vacantes ofrecidas"),
    kpi(mediana_salario, "salario mediano mensual bruto", prefijo="B/. "),
    kpi(float(limpio["experiencia_meses"].median()), "meses de experiencia, mediana de los que la piden"),
])

transparencia = "".join([
    kpi(publica_confidencial["Empresa identificada"], "de las empresas identificadas publica salario", sufijo="%", decimales=1),
    kpi(publica_confidencial["Confidencial"], "de las confidenciales publica salario", sufijo="%", decimales=1),
    kpi(pct_semi_o_sin_nivel * 100, "de los salarios viene de avisos Semi Sr o sin nivel", sufijo="%"),
])

modalidades = "".join(kpi(presencial.get(m, 0), m.lower(), sufijo="%", decimales=1) for m in ["Presencial", "Híbrido", "Remoto"])

# Portada: cada hecho se expresa «de cada 100 avisos»
def de_cada_100(fraccion):
    """Convierte una fracción en un número entero de cuadros de la cuadrícula."""
    return int(round(float(fraccion) * 100))

n_salario = de_cada_100(limpio["publica_salario"].mean())
n_panama = de_cada_100(pct_panama / 100)
n_presencial = de_cada_100(presencial["Presencial"] / 100)
n_confidencial = de_cada_100(limpio["empresa_confidencial"].mean())
n_experiencia = de_cada_100(limpio["pide_experiencia"].mean())
n_edad = de_cada_100(limpio["fija_edad"].mean())
n_ingles = de_cada_100(limpio["pide_ingles"].mean())

hechos = [
    {"boton": "Publican salario", "cifra": n_salario, "frase": "dicen cuánto pagan. El resto deja el salario en silencio."},
    {"boton": f"En {provincia_top['provincia']}", "cifra": n_panama, "frase": f"están en la provincia de {provincia_top['provincia']}. Las demás se reparten lo que queda."},
    {"boton": "Presenciales", "cifra": n_presencial, "frase": "son presenciales. El trabajo flexible casi no aparece."},
    {"boton": "Confidenciales", "cifra": n_confidencial, "frase": "ocultan el nombre de la empresa."},
    {"boton": "Piden experiencia", "cifra": n_experiencia, "frase": "piden experiencia de forma explícita."},
    {"boton": "Fijan edad", "cifra": n_edad, "frase": "fijan un rango de edad."},
    {"boton": "Piden inglés", "cifra": n_ingles, "frase": "piden inglés."},
]

# Datos para las partes interactivas (portada y explorador)
perfil_general = aux.assign(no_presencial=aux["modalidad"] != "Presencial")
datos_pagina = {
    "hechos": hechos,
    "areas": json.loads(t10_perfil_area.to_json(orient="records", force_ascii=False)),
    "general": {
        "pct_publica_salario": float(limpio["publica_salario"].mean() * 100),
        "pct_pide_experiencia": float(limpio["pide_experiencia"].mean() * 100),
        "pct_universitario_o_mas": float(perfil_general["universitario_o_mas"].mean() * 100),
        "pct_pide_ingles": float(limpio["pide_ingles"].mean() * 100),
        "pct_hibrido_o_remoto": float(perfil_general["no_presencial"].mean() * 100),
    },
}

### 8.5 Pestaña de análisis exploratorio
Reúne lo que la sección 7 guardó en `eda`, además de los hallazgos y el contraste con la propuesta. Los gráficos y los textos no se repiten a mano: si cambian los datos, esta pestaña cambia con ellos.

In [351]:
SECCIONES_EDA = [
    ("7.1", "Perfil antes y después", "¿Qué cambió con la limpieza?"),
    ("7.2", "Descriptivos numéricos", "¿Cuáles son los valores típicos de las variables numéricas?"),
    ("7.3", "Frecuencias", "¿Qué valores se repiten más en las variables categóricas?"),
    ("7.4", "Distribuciones", "¿Cómo se reparten el salario y la experiencia?"),
    ("7.5", "Comparaciones", "¿En qué se diferencian las áreas y los niveles?"),
    ("7.6", "Relaciones", "¿Qué variables se mueven juntas?"),
    ("7.7", "Tendencia", "¿Cómo cambia la publicación en el tiempo?"),
    ("7.8", "Hallazgos", "¿Qué se encontró, con la cifra que lo sustenta?"),
    ("7.9", "Contraste", "¿Se cumplió lo esperado en la propuesta?"),
]

def con_negritas(texto):
    """Escapa el texto y convierte el **resaltado** de Markdown en HTML."""
    return re.sub(r"\*\*(.+?)\*\*", r"<strong>\1</strong>", escape(texto))

def tabla_eda(tabla):
    """Tabla visible; el índice solo se muestra si tiene nombre, porque entonces es una columna con significado."""
    return f'<div class="tabla tabla-eda">{tabla.to_html(index=bool(tabla.index.name), border=0, na_rep="—")}</div>'

def tarjeta_eda(pieza):
    """Tarjeta de la pestaña: gráfico o tablas de la sección 7, con su interpretación."""
    titulo, grafico = "", ""
    if pieza["fig"] is not None:
        fig = go.Figure(pieza["fig"])                 # copia: el gráfico de la sección 7 no cambia
        titulo = f'<h3>{escape(fig.layout.title.text)}</h3>'
        fig.update_layout(title=None)
        for i, traza in enumerate(fig.data):
            color = [AZUL, NARANJA, AQUA, AMARILLO][i % 4]
            if traza.type == "scatter":
                traza.line.color = color
            if traza.type != "heatmap":
                traza.marker.color = color
        estilo_grafico(fig, fig.layout.height or 420, leyenda=len(fig.data) > 1)
        grafico = fig.to_html(full_html=False, include_plotlyjs=False, config={"displayModeBar": False, "responsive": True})
    tablas = "".join(tabla_eda(t) for t in pieza["tablas"])
    return f'<section class="tarjeta aparece">{titulo}{grafico}<p class="interpretacion">{con_negritas(pieza["texto"])}</p>{tablas}</section>'

lista_hallazgos = "".join(f"<li>{con_negritas(h)}</li>" for h in hallazgos)
contenido_eda = {n: [tarjeta_eda(p) for p in eda.values() if p["seccion"] == n] for n, _, _ in SECCIONES_EDA}
contenido_eda["7.8"] = [f'<section class="tarjeta aparece"><ol class="lista-hallazgos">{lista_hallazgos}</ol></section>']
contenido_eda["7.9"] = [f'<section class="tarjeta aparece">{tabla_eda(contraste)}</section>']

secciones_eda = [capitulo(n, f'eda-{n.replace(".", "-")}', t, p, contenido_eda[n]) for n, t, p in SECCIONES_EDA]


### 8.6 Escritura de la página
Los estilos (`estilos.css`) y el comportamiento de la página (`pagina.js`) están en archivos aparte en `reporte/dashboard/plantillas/`; se leen y se incrustan en el HTML.

Une las piezas en un solo HTML y lo guarda en `reporte/dashboard/index.html`, que cada ejecución reemplaza. También guarda una copia con la fecha y la hora en el nombre (`index_AAAAMMDD_HHMMSS.html`) para conservar cada versión.

In [352]:
# Estilos y comportamiento de la página: archivos aparte en reporte/dashboard/plantillas/ que se incrustan en index.html,
# así el dashboard sigue siendo un solo archivo que funciona sin internet
ESTILOS = (DIR_DASHBOARD / "plantillas" / "estilos.css").read_text(encoding="utf-8")
PAGINA_JS = (DIR_DASHBOARD / "plantillas" / "pagina.js").read_text(encoding="utf-8")

In [353]:
# Tablas que se muestran debajo de cada gráfico
columnas_salario = ["area", "con_salario", "pct_publica_salario", "salario_q1", "salario_mediana", "salario_q3"]
columnas_requisitos = ["nivel_laboral", "avisos", "pct_universitario_o_mas", "pct_pide_experiencia", "pct_fija_edad", "pct_pide_ingles"]
semanal_vista = t7_semanal.assign(semana=t7_semanal["semana"].dt.strftime("%d/%m/%Y"))

nota_mapa = "Esquema de ubicación, no a escala."
if len(fuera_del_mapa):
    nota_mapa += " Fuera del mapa: " + ", ".join(f"«{f['provincia']}» ({f['avisos']})" for _, f in fuera_del_mapa.iterrows()) + "."

capitulos = [
    capitulo("1", "donde", "¿Dónde está la demanda?", "Primero, qué áreas y qué regiones concentran los avisos.", [
        tarjeta(f"3 áreas reúnen el {top3_areas}% de los avisos", fig_area,
                f"{area_top['area']} lidera con {area_top['pct_avisos']}% de los avisos y {area_top['pct_vacantes']}% de las vacantes.",
                t1_demanda_area.head(10), f"Las 10 áreas con más avisos, de {len(t1_demanda_area)}. n = {len(limpio):,} avisos."),
        tarjeta(f"{industria_top['industria']} es la industria con más avisos: {industria_top['pct_avisos']}% de los que la indican", fig_industria,
                f"Solo el {len(con_industria) / len(limpio):.0%} de los avisos indica su industria, y el {limpio.loc[limpio['industria'] == 'Sin dato', 'empresa_confidencial'].mean():.0%} de los que no lo hacen son confidenciales: esta vista no cubre todo el mercado.",
                industrias_grafico, f"Las 10 industrias con más avisos, sin contar la categoría genérica «Otra». n = {len(con_industria):,} avisos con industria conocida."),
        f'''<section class="tarjeta aparece"><h3>La provincia de {escape(str(provincia_top["provincia"]))} concentra el {pct_panama}% de los avisos</h3>
        {mapa_provincias(t2_demanda_provincia)}{leyenda_mapa}
        <p class="interpretacion">La sigue {escape(str(provincia_2["provincia"]))} con {provincia_2["pct_avisos"]}%: la demanda formal en línea se concentra en un solo polo.</p>
        {nota_y_datos(f"{nota_mapa} Cifras: n.º de avisos y % del total. n = {len(limpio):,} avisos.", t2_demanda_provincia)}</section>''',
    ]),
    capitulo("2", "ofrecen", "¿Qué ofrecen las empresas?", "Sabiendo dónde está la demanda, la pregunta es cuánto se paga. Lo primero que aparece es que casi nadie lo dice.", [
        f'<div class="kpis aparece">{transparencia}</div>',
        tarjeta(f"{mas_alta['area']} paga la mediana más alta: B/. {mas_alta['salario_mediana']:,.0f}", fig_salario,
                f"Entre las áreas comparables, la mediana va de B/. {mas_baja['salario_mediana']:,.0f} ({mas_baja['area']}) a B/. {mas_alta['salario_mediana']:,.0f}: {mas_alta['salario_mediana'] / mas_baja['salario_mediana']:.1f} veces más.",
                areas_comparables[columnas_salario],
                f"Punto: mediana. Línea: del primer al tercer cuartil. Solo áreas con al menos {MIN_AVISOS_SALARIO} salarios. n = {len(datos_area)} avisos con salario."),
    ]),
    capitulo("3", "piden", "¿Qué piden las empresas?", "Del lado de la oferta se sabe poco. Del lado de los requisitos, el patrón por nivel es claro.", [
        tarjeta(f"La exigencia sube con el nivel: universitario de {req_bajo['pct_universitario_o_mas']:.0f}% a {req_alto['pct_universitario_o_mas']:.0f}%", fig_requisitos,
                f"De {nombre_bajo} a {nombre_alto}, la experiencia mediana pasa de {req_bajo['experiencia_mediana_meses']:.0f} a {req_alto['experiencia_mediana_meses']:.0f} meses y el inglés de {req_bajo['pct_pide_ingles']:.0f}% a {req_alto['pct_pide_ingles']:.0f}%.",
                requisitos_grafico[columnas_requisitos],
                f"Niveles jerárquicos con al menos {MIN_AVISOS_AREA} avisos. n = {requisitos_grafico['avisos'].sum():,} avisos.",
                div_id="grafico-requisitos"),
    ]),
    capitulo("4", "cuando", "¿Cuándo y cómo se contrata?", "Por último, cómo evoluciona la publicación y si el trabajo flexible tiene peso.", [
        tarjeta(f"{MESES[mes_final].capitalize()} promedia {por_mes[mes_final]:.0f} avisos por semana, frente a {por_mes[mes_inicial]:.0f} en {MESES[mes_inicial]}", fig_semanal,
                f"Entre las semanas comparables, la publicación cambia {(por_mes[mes_final] / por_mes[mes_inicial] - 1):+.0%}. Las semanas anteriores al {inicio_comparable:%d/%m} no indican menos demanda: esos avisos ya habían vencido.",
                semanal_vista, f"n = {t7_semanal['avisos'].sum():,} avisos."),
        f'<div class="kpis aparece">{modalidades}</div>',
        tarjeta(f"El trabajo flexible apenas existe: máximo {modalidad_top['pct_hibrido_o_remoto']}% en {modalidad_top['area']}", fig_modalidad,
                f"En conjunto, híbrido y remoto suman {presencial.drop('Presencial').sum():.1f}% de los avisos.",
                modalidad_grafico, f"Áreas con al menos {MIN_AVISOS_AREA} avisos y algún aviso híbrido o remoto."),
    ]),
    capitulo("5", "tu-area", "¿Cómo es tu área?", "Elige un área y compárala con el mercado completo.", [
        f'''<section class="tarjeta aparece explorador"><div>
        <label for="selector-area">Área</label><select id="selector-area"></select>
        <div class="datos-area">
        <div><strong id="area-avisos"></strong><span>avisos</span></div>
        <div><strong id="area-vacantes"></strong><span>vacantes</span></div>
        <div><strong id="area-puesto"></strong><span>puesto por avisos</span></div>
        <div><strong id="area-salario"></strong><span>salario mediano mensual</span></div>
        </div></div>
        <div><div id="filas-area"></div>
        <p class="nota">Barra: el área elegida. Línea vertical: el mercado completo. % de los avisos del área. La mediana salarial solo aparece con al menos {MIN_AVISOS_SALARIO} salarios publicados.</p>
        {nota_y_datos("", t10_perfil_area)}</div></section>''',
    ]),
]

primero = hechos[0]
cuadros_html = "".join(f'<i style="--i:{i}" class="{"lleno" if i < primero["cifra"] else ""}"></i>' for i in range(100))
botones_html = "".join(f'<button type="button" aria-pressed="{"true" if i == 0 else "false"}">{escape(h["boton"])}</button>' for i, h in enumerate(hechos))
enlaces_html = "".join(f'<a href="#{a}">{t}</a>' for a, t in [("donde", "Dónde"), ("ofrecen", "Qué ofrecen"), ("piden", "Qué piden"),
                                                              ("cuando", "Cuándo y cómo"), ("tu-area", "Tu área")])
enlaces_eda_html = "".join(f'<a href="#eda-{n.replace(".", "-")}">{n} {t.split()[0]}</a>' for n, t, _ in SECCIONES_EDA)
datos_json = json.dumps(datos_pagina, ensure_ascii=False).replace("</", "<\\/")
ahora = datetime.now()
generado = ahora.strftime("%d/%m/%Y %H:%M")

pagina = f"""<!doctype html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<meta name="author" content="Simón Ali Bustamante Blanco">
<!-- Hecho por Simón Ali Bustamante Blanco · Proyecto final IPCD · UIP · 2026 -->
<title>Demanda laboral en Panamá</title>
<style>{ESTILOS}</style>
<script>{get_plotlyjs()}</script>
</head>
<body>
<div id="progreso"></div>
<header class="hero"><div class="hero-interior">
<div>
<p class="antetitulo">Mercado laboral · {inicio_periodo:%d/%m/%Y} al {fin_periodo:%d/%m/%Y}</p>
<h1>¿Qué piden y qué ofrecen las empresas en Panamá?</h1>
<p class="hero-lead">{len(limpio):,} avisos de empleo de Konzerta, limpiados y analizados. Esta es la historia del mercado contada en 100 avisos.</p>
</div>
<div class="panel-100">
<div class="cabeza-100"><strong id="cifra">{primero["cifra"]}</strong><span>de cada 100 avisos</span></div>
<p id="frase" aria-live="polite">{escape(primero["frase"])}</p>
<div id="reticula" class="reticula" role="img" aria-label="{primero["cifra"]} de cada 100 avisos: {escape(primero["frase"])}">{cuadros_html}</div>
<div id="hechos" class="botones">{botones_html}</div>
</div>
</div></header>
<nav class="indice" aria-label="Navegación">
<div class="pestanas"><button type="button" data-abre="historia" aria-pressed="true">Historia</button><button type="button" data-abre="analisis" aria-pressed="false">Análisis exploratorio</button></div>
<div data-vista="historia">{enlaces_html}</div>
<div data-vista="analisis" hidden>{enlaces_eda_html}</div>
</nav>
<main data-vista="historia">
<div class="kpis flotante aparece">{indicadores}</div>
{"".join(capitulos)}
</main>
<main data-vista="analisis" hidden>
<p class="pregunta entrada-eda">El análisis completo detrás de la historia: las mismas cifras de la sección 7 del notebook, calculadas desde los datos limpios.</p>
{"".join(secciones_eda)}
</main>
<footer>Simón Ali Bustamante Blanco · Introducción a la Programación para Ciencia de Datos · UIP. Datos: avisos públicos de konzerta.com, extraídos el {fecha_extraccion:%d/%m/%Y}. Las empresas confidenciales no se identifican. Generado el {generado} desde {escape(ARCHIVO_ORIGINAL.name)}.</footer>
<script>const DATOS = {datos_json};</script>
<script>{PAGINA_JS}</script>
</body>
</html>"""

DIR_DASHBOARD.mkdir(parents=True, exist_ok=True)
ARCHIVO_DASHBOARD = DIR_DASHBOARD / "index.html"
ARCHIVO_DASHBOARD.write_text(pagina, encoding="utf-8")
ARCHIVO_VERSION = DIR_DASHBOARD / f"index_{ahora:%Y%m%d_%H%M%S}.html"   # copia que no se reemplaza
ARCHIVO_VERSION.write_text(pagina, encoding="utf-8")

print(f"Guardado: {ARCHIVO_DASHBOARD.relative_to(BASE)} ({ARCHIVO_DASHBOARD.stat().st_size / 1_000_000:.1f} MB)")
print(f"Copia con fecha y hora: {ARCHIVO_VERSION.relative_to(BASE)}")
print(f"Gráficos Plotly: {pagina.count('plotly-graph-div')}")
print(f"Piezas del EDA en la pestaña: {len(eda)}")
print("Enlaces a avisos en la página:", pagina.count("konzerta.com/empleos"))

Guardado: reporte/dashboard/index.html (5.1 MB)
Copia con fecha y hora: reporte/dashboard/index_20261105_212433.html
Gráficos Plotly: 19
Piezas del EDA en la pestaña: 14
Enlaces a avisos en la página: 0
